# Phase 2: Tensor-Ready Dataset (Post-Merge)

This notebook starts the tensor-ready pipeline after Phase 1 cleaning is complete. It loads the final cleaned events, prepares placeholders for stints and tracking merges, and sets up the tensor-ready encoding steps.

## 1. Load Existing Notebook and Set Paths

Define input and output paths used throughout the pipeline.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

BASE_DIR = Path(r"x:\My Files\Python\Sports Analytics Projects\Hockey\HALO")
DATA_DIR = BASE_DIR / "HALO Hackathon Data"
INUPT_DIR = BASE_DIR / "Processed Sequence Data"
OUTPUT_DIR = BASE_DIR / "Tensor-Ready Data"


print(f"BASE_DIR: {BASE_DIR}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"INUPT_DIR: {INUPT_DIR}")

BASE_DIR: x:\My Files\Python\Sports Analytics Projects\Hockey\HALO
DATA_DIR: x:\My Files\Python\Sports Analytics Projects\Hockey\HALO\HALO Hackathon Data
OUTPUT_DIR: x:\My Files\Python\Sports Analytics Projects\Hockey\HALO\Tensor-Ready Data
INUPT_DIR: x:\My Files\Python\Sports Analytics Projects\Hockey\HALO\Processed Sequence Data


## 2. Finalize EDA Checks and Verification Cells

Quick sanity checks on the finalized cleaned dataset.

In [2]:
print("=== Quick EDA + Phase 2 Prep ===")

final_path = INUPT_DIR / "events_clean_final.parquet"
if not final_path.exists():
    raise FileNotFoundError(f"Missing file: {final_path}")

df_events = pd.read_parquet(final_path)
print(f"Rows: {len(df_events):,}")
print(f"Games: {df_events['game_id'].nunique():,}")
print(f"Sequences: {df_events.groupby(['game_id', 'sequence_id']).ngroups:,}")
print(f"Goals: {(df_events['event_type'] == 'goal').sum():,}")

if "sl_event_id" not in df_events.columns:
    raise KeyError("Missing required column: sl_event_id")

def normalize_text(value: object) -> object:
    if pd.isna(value):
        return value
    text = str(value).replace("+", " ").replace("-", " ")
    text = " ".join(text.split())
    return text.lower()

if "description" in df_events.columns:
    df_events["description_clean"] = df_events["description"].apply(normalize_text)
else:
    df_events["description_clean"] = np.nan

if "flags" in df_events.columns:
    df_events["flags_clean"] = df_events["flags"].apply(normalize_text)
else:
    df_events["flags_clean"] = np.nan

# Placeholder for manual synonym normalization
synonym_map = {
    # "old term": "new term",
}
if synonym_map:
    df_events["description_clean"] = df_events["description_clean"].replace(synonym_map)
    df_events["flags_clean"] = df_events["flags_clean"].replace(synonym_map)

keep_cols = [
    "game_id",
    "period",
    "period_time",
    "game_stint",
    "sequence_id",
    "player_id",
    "team_id",
    "opp_team_id",
    "event_type",
    "outcome",
    "flags",
    "flags_clean",
    "description",
    "description_clean",
    "detail",
    "x",
    "y",
    "x_adj",
    "y_adj",
    "has_tracking_data",
    "event_player_tracked",
    "dest_x_adj",
    "dest_y_adj",
    "sl_event_id",
    "sl_xg_all_shots",
 ]
available_cols = [col for col in keep_cols if col in df_events.columns]
missing_cols = sorted(set(keep_cols) - set(available_cols))
if missing_cols:
    print("Missing columns (not found in events):", missing_cols)

df_events = df_events[available_cols].copy()
print("Columns retained:", len(df_events.columns))
print("Unique cleaned descriptions:", df_events["description_clean"].nunique(dropna=True))
print("Unique cleaned flags fields:", df_events["flags_clean"].nunique(dropna=True))
print(df_events["description_clean"].value_counts(dropna=False).head(10))

=== Quick EDA + Phase 2 Prep ===
Rows: 1,658,412
Games: 480
Sequences: 28,095
Goals: 2,810
Columns retained: 25
Unique cleaned descriptions: 358
Unique cleaned flags fields: 3659
description_clean
o zone pass reception      119461
d zone pass reception      109196
n zone pass reception       59785
shot pressure               53961
block opposition pass       47654
dz outlet pass              45828
dzone d2d                   42200
carry over redline          41030
controlled exit from dz     37539
dumpin against              32182
Name: count, dtype: int64


In [3]:
print("\n=== Description + Flags Cleaning (Leakage-Safe for Embeddings) ===")

import re

if "description_clean" not in df_events.columns:
    raise KeyError("Missing description_clean. Run the EDA cell first.")
if "flags_clean" not in df_events.columns:
    raise KeyError("Missing flags_clean. Run the EDA cell first.")

# -----------------------------
# Description normalization (in-place -> description_clean)
# -----------------------------
abbrev_patterns = [
    (r"\b(f/off|face\s*off|faceoff)\b", "faceoff"),
    (r"\bchk\b", "check"),
    (r"\brec\b", "reception"),
    (r"\bdef\b", "defence"),
    (r"\bblue hold\b", "blueline hold"),
    (r"\b(e\s*w|e/w)\b", "east/west"),
]
for pattern, replacement in abbrev_patterns:
    df_events["description_clean"] = df_events["description_clean"].str.replace(
        pattern, replacement, regex=True
    )

zone_patterns = [
    (r"\b(dz|d[- ]?zone|dzone|defensive zone)\b", "defensive zone"),
    (r"\b(nz|n[- ]?zone|nzone|neutral zone)\b", "neutral zone"),
    (r"\b(oz|o[- ]?zone|ozone|offensive zone)\b", "offensive zone"),
    (r"\b(dzpass|d[- ]?zone\s*pass|defensive zone\s*pass)\b", "defensive zone pass"),
    (r"\b(nzpass|n[- ]?zone\s*pass|neutral zone\s*pass)\b", "neutral zone pass"),
    (r"\b(ozpass|o[- ]?zone\s*pass|offensive zone\s*pass)\b", "offensive zone pass"),
    (r"\b(dzdeke|d[- ]?zone\s*deke|defensive zone\s*deke)\b", "defensive zone deke"),
    (r"\b(nzdeke|n[- ]?zone\s*deke|neutral zone\s*deke)\b", "neutral zone deke"),
    (r"\b(ozdeke|o[- ]?zone\s*deke|offensive zone\s*deke)\b", "offensive zone deke"),
]
for pattern, replacement in zone_patterns:
    df_events["description_clean"] = df_events["description_clean"].str.replace(
        pattern, replacement, regex=True
    )

replacements = {
    r"\blpr\b": "loose puck recovery",
    r"\bcont\b": "contested",
    r"\breb\b": "rebound",
    r"\bopdump\b": "opposing dump",
    r"\bhipresopdump\b": "high pressure opposing dump",
    r"\bnofore\b": "no forecheck",
    r"\bhi press\b": "high pressure",
    r"offboards": "off boards",
    r"\bd2d\b": "defenceman to defenceman pass",
    r"\bbc\b": "board cycle",
    r"\bsc\b": "slot cycle",
    r"\bci\b": "chip in",
    r"\bdi\b": "dump in",
    "nzpuckprotection": "neutral zone puck protection",
    "ozpuckprotection": "offensive zone puck protection",
    "dzpuckprotection": "defensive zone puck protection",
    "lprreb": "loose puck recovery rebound",
    "failed pass trajectory location": "failed pass to",
    "offensive zone entry pass reception": "offensive zone entry pass successful",
    "defensive deflection; original description: outside shot for onnet": "defensive deflection",
    "defensive deflection; original description: slot shot for onnet": "defensive deflection",
    r"\bnz\b": "neutral zone",
    r"\boz\b": "offensive zone",
    r"\bdz\b": "defensive zone",
}
for old, new in replacements.items():
    df_events["description_clean"] = df_events["description_clean"].str.replace(
        old, new, regex=True, case=False
    )

# -----------------------------
# Flags cleaning and tokenization (in-place -> flags_clean)
# -----------------------------
def normalize_flag_token(token: str) -> str:
    text = str(token).lower()
    text = text.replace("+", " ").replace("-", " ")

    # Remove any brackets
    text = re.sub(r"[\[\]\(\)\{\}]", "", text)

    # Remove known source prefixes at token start
    text = re.sub(r"^(?:shot|deflection)\s*_?\s*flags?\s*[:_\-\s]*", "", text)

    # Normalize leftover punctuation/separators
    text = re.sub(r"[;|/]", " ", text)
    text = re.sub(r"\s+", " ", text).strip(" ,._:-")
    return text

def split_clean_flags(value: object) -> list[str]:
    if pd.isna(value):
        return []

    text = str(value).lower()
    text = text.replace("+", " ").replace("-", " ")
    text = re.sub(r"[\[\]\(\)\{\}]", "", text)
    text = re.sub(r"\s*[,;|/]\s*", ",", text)
    text = re.sub(r"\s+", " ", text).strip()
    if not text:
        return []

    raw_tokens = [tok.strip() for tok in text.split(",") if tok.strip()]
    clean_tokens = []
    for tok in raw_tokens:
        tok_clean = normalize_flag_token(tok)
        if tok_clean:
            clean_tokens.append(tok_clean)

    # De-duplicate while preserving order
    return list(dict.fromkeys(clean_tokens))

# -----------------------------
# Leakage removal (in-place -> description_clean and flags_clean)
# -----------------------------
leak_regex = re.compile(
    r"\bgoal\b|\bgoals\b|\bon\s*net\b|\bonnet\b|\bwith\s*goal\b|\bwith\s*shot\s*on\s*net(?:\s*whistle)?\b",
    flags=re.IGNORECASE
)

blocked_compact_tokens = {
    "withgoal",
    "withshotonnet",
    "withshotonnetwhistle",
}

def remove_leak_terms(text: object) -> object:
    if pd.isna(text):
        return text
    cleaned = leak_regex.sub(" ", str(text).lower())
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return cleaned if cleaned else np.nan

def keep_non_leak_flag_token(tok: str) -> bool:
    compact = re.sub(r"[^a-z0-9]", "", str(tok).lower())
    if compact in blocked_compact_tokens:
        return False
    if leak_regex.search(str(tok)):
        return False
    return True

# Clean descriptions in-place
df_events["description_clean"] = df_events["description_clean"].apply(remove_leak_terms)

# Clean flags in-place via temporary local token list (not stored as extra columns)
flags_tokens_final = df_events["flags_clean"].apply(split_clean_flags).apply(
    lambda toks: [tok for tok in toks if keep_non_leak_flag_token(tok)]
)
df_events["flags_clean"] = flags_tokens_final.apply(
    lambda toks: ", ".join(toks) if len(toks) > 0 else np.nan
)


=== Description + Flags Cleaning (Leakage-Safe for Embeddings) ===


In [4]:
print("\n=== Final flags_clean trimming ===")
 
drop_tokens = {
    "gwg",
    "highdangermissedshotrecovery",
    "fivehole",
    "deflected",
    "fanned",
    "firstafterpp",
    "whistlebeforeexit",
    "withcyclepass",
    "withfailedcyclepass",
    "withplayafter",
    "withpressure",
    "withrebound",
    "withslotshot"
}
 
def count_tokens(value: object) -> int:
    if pd.isna(value):
        return 0
    return len([tok for tok in str(value).split(",") if tok.strip()])
 
def final_flags_cleanup(value: object) -> object:
    if pd.isna(value):
        return value
    parts = [tok.strip() for tok in str(value).split(",") if tok.strip()]
    filtered = []
    for tok in parts:
        tok_clean = tok.lower()
        if "*" in tok_clean:
            continue
        if tok_clean in drop_tokens:
            continue
        filtered.append(tok_clean)
    filtered = list(dict.fromkeys(filtered))
    return ", ".join(filtered) if filtered else np.nan
 
pre_token_counts = df_events["flags_clean"].apply(count_tokens)
df_events["flags_clean"] = df_events["flags_clean"].apply(final_flags_cleanup)
post_token_counts = df_events["flags_clean"].apply(count_tokens)
 
print("Tokens before trimming:", pre_token_counts.sum())
print("Tokens after trimming:", post_token_counts.sum())
print("Rows affected:", (post_token_counts != pre_token_counts).sum())
 
df_events["embedding_text_clean"] = (
    df_events["description_clean"].fillna("") +
    " ; " +
    df_events["flags_clean"].fillna("")
).str.strip(" ;")
df_events.loc[df_events["embedding_text_clean"] == "", "embedding_text_clean"] = np.nan


=== Final flags_clean trimming ===
Tokens before trimming: 1213509
Tokens after trimming: 1097347
Rows affected: 69768


In [5]:
# -----------------------------
# Audits requested: unique flags + leakage phrase checks
# -----------------------------
def _split_final_flags(value: object) -> list[str]:
    if pd.isna(value):
        return []
    return [tok.strip() for tok in str(value).split(",") if tok.strip()]

final_flags_series = df_events["flags_clean"].apply(_split_final_flags)
all_flag_tokens = sorted(set(tok for toks in final_flags_series for tok in toks))
flag_leak_tokens = [tok for tok in all_flag_tokens if (not keep_non_leak_flag_token(tok))]

print(f"Total unique cleaned flag tokens: {len(all_flag_tokens):,}")
print("Top 30 flag tokens:")
print(pd.Series(all_flag_tokens).head(30).to_string(index=False))

unique_flags_path = OUTPUT_DIR / "unique_flags_clean_tokens.csv"
pd.DataFrame({"flag_token_clean": all_flag_tokens}).to_csv(unique_flags_path, index=False)
print(f"Saved full unique flag token list: {unique_flags_path}")

print("\nLeakage token hits inside cleaned flags (goal/on net/onnet/withgoal/withshotonnet*):")
if len(flag_leak_tokens) == 0:
    print("  None")
else:
    print(pd.Series(flag_leak_tokens).to_string(index=False))

desc_hits_after = df_events["description_clean"].astype(str).str.contains(leak_regex, na=False).sum()
flags_hits_after = df_events["flags_clean"].astype(str).str.contains(leak_regex, na=False).sum()
print(f"\nLeakage phrase hits AFTER removal:")
print(f"  description_clean: {desc_hits_after:,}")
print(f"  flags_clean: {flags_hits_after:,}")

# Inspect any misses after removal (if any)
if desc_hits_after > 0:
    print("\nSample remaining description hits:")
    print(df_events.loc[
        df_events["description_clean"].astype(str).str.contains(leak_regex, na=False),
        ["description", "description_clean"]
    ].head(10).to_string(index=False))

if flags_hits_after > 0:
    print("\nSample remaining flag hits:")
    print(df_events.loc[
        df_events["flags_clean"].astype(str).str.contains(leak_regex, na=False),
        ["flags", "flags_clean"]
    ].head(10).to_string(index=False))

unique_desc = (
    df_events["description_clean"]
    .dropna()
    .astype(str)
    .drop_duplicates()
    .sort_values()
 )
unique_desc_path = OUTPUT_DIR / "description_unique_clean.csv"
unique_desc.to_csv(unique_desc_path, index=False, header=["description_clean"] )
print("Saved unique descriptions:", unique_desc_path)

Total unique cleaned flag tokens: 39
Top 30 flag tokens:
             1timer
        alongboards
           backhand
   backhandforehand
        centerfodot
              crest
           crossice
          eastfodot
          eastofdot
           eastside
           eastwest
   firstozreception
   forehandbackhand
           highleft
          highright
 lefthandedopponent
            lowhigh
            lowleft
           lowright
             midice
         nzgreyarea
       quickrelease
          recovered
righthandedopponent
                rim
           screened
               seam
           slapshot
           snapshot
               soft
Saved full unique flag token list: x:\My Files\Python\Sports Analytics Projects\Hockey\HALO\Tensor-Ready Data\unique_flags_clean_tokens.csv

Leakage token hits inside cleaned flags (goal/on net/onnet/withgoal/withshotonnet*):
  None

Leakage phrase hits AFTER removal:
  description_clean: 0
  flags_clean: 0
Saved unique descriptions: x:\My F

In [6]:
print("\n=== Description Word Frequency ===")

if "description_clean" not in df_events.columns:
    raise KeyError("Missing description_clean. Run the prior cleaning cells first.")

word_series = (
    df_events["description_clean"]
    .dropna()
    .astype(str)
    .str.split()
    .explode()
    .str.strip()
)
word_series = word_series[word_series != ""]

word_counts = word_series.value_counts()
print("Top 50 words:")
print(word_counts.head(50))


=== Description Word Frequency ===
Top 50 words:
description_clean
zone          1131598
pass           725486
defensive      503283
offensive      433703
puck           349318
loose          338840
recovery       338836
reception      317322
neutral        199904
to             198034
off            186338
defenceman     148972
dump           138323
shot           136134
cycle          127394
controlled     112369
boards         112228
failed          97744
|               97357
entry           90006
contested       87589
faceoff         84261
north           82729
outlet          81377
in              81030
opposition      76917
pressure        74488
against         70263
south           69322
block           61474
slot            58606
for             55993
from            50023
deke            44523
exit            42769
over            41204
redline         41030
carry           41030
defence         39532
out             35152
east/west       34325
outside         33517
dumpin  

In [7]:
df_events.head()

,game_id,period,period_time,game_stint,sequence_id,player_id,team_id,opp_team_id,event_type,outcome,...,y,x_adj,y_adj,has_tracking_data,event_player_tracked,dest_x_adj,dest_y_adj,sl_event_id,sl_xg_all_shots,embedding_text_clean
0,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,0E-9,1.0,1,8cdcb61e-d733-bde6-a101-ef3140e48149,6cac12e2-0546-2c1a-689f-ab26d8a6355a,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,faceoff,failed,...,-0.755550,-0.201431,-0.755550,1,1,NaN,NaN,1,NaN,"neutral zone faceoff ; centerfodot, righthande..."
1,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,0E-9,1.0,1,dc5a9c10-c8d3-52bc-fd80-b7186f383b34,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,6cac12e2-0546-2c1a-689f-ab26d8a6355a,faceoff,successful,...,-0.755550,0.201431,0.755550,1,1,NaN,NaN,2,NaN,neutral zone reception faceoff entry ; centerf...
2,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,0.070000000,1.0,1,dc5a9c10-c8d3-52bc-fd80-b7186f383b34,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,6cac12e2-0546-2c1a-689f-ab26d8a6355a,lpr,successful,...,1.257381,-0.807243,-1.257381,1,1,NaN,NaN,3,NaN,faceoff loose puck recovery neutral zone
3,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,0.130000000,1.0,1,dc5a9c10-c8d3-52bc-fd80-b7186f383b34,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,6cac12e2-0546-2c1a-689f-ab26d8a6355a,pass,successful,...,0.845894,-0.304306,-0.845894,1,1,NaN,NaN,4,NaN,neutral zone pass south ; eastside
4,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,2.200000000,1.0,1,0753b094-9e2f-976d-85c8-d22a1d280e8d,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,6cac12e2-0546-2c1a-689f-ab26d8a6355a,reception,successful,...,20.917810,-6.339600,-20.917810,1,1,NaN,NaN,5,NaN,neutral zone pass reception


In [8]:
# Save cleaned events to parquet for next phase
events_path = OUTPUT_DIR / "events_clean_description_tensor.parquet"
df_events.to_parquet(events_path)


## 3. Stints Data

Load stints and prepare context features (skaters, score differential, power play, empty net).

In [3]:
print("\n=== Stints Merge ===")

#Load events if not already loaded
events_path = OUTPUT_DIR / "events_clean_description_tensor.parquet"
df_events = pd.read_parquet(events_path)
print(f"Loaded events: {len(df_events):,}")

# 1) Load stints
stints_path = DATA_DIR / "stints.parquet"
if not stints_path.exists():
    print(f"Missing stints file: {stints_path}")
else:
    df_stints = pd.read_parquet(stints_path)
    print(f"Loaded stints: {len(df_stints):,}")
    
keep_cols = ["game_id", "period", "period_time_start", "period_time_end", 
             "game_stint", "n_home_skaters", "n_away_skaters", 
             "is_home_net_empty", "is_away_net_empty",
             "home_score", "away_score"]

df_stints.drop(columns=[col for col in df_stints.columns if col not in keep_cols], inplace=True) 

#Remove duplicate stint records if they exist (keep first occurrence)
df_stints = df_stints.drop_duplicates(subset=["game_id", "period", "game_stint"], keep="first")

print("Stints columns:", df_stints.columns.tolist())
print("Stints total rows:", len(df_stints))
print("Stints sample:")
print(df_stints.head())

merge_keys = ["game_id", "period", "game_stint"]

# Merge (left join, one record per stint)
df_events = df_events.merge(
        df_stints,
        on=merge_keys,
        how="left",
        validate="m:1"
    )

if "home_score" in df_events.columns and "away_score" in df_events.columns:
        df_events["score_differential_home"] = df_events["home_score"] - df_events["away_score"]
        df_events["score_differential_away"] = df_events["away_score"] - df_events["home_score"]

print("Stints merge complete.")
print("Missing stints rows:", df_events["home_score"].isna().sum())

df_events.head()

# Save merged events for next phase
merged_path = OUTPUT_DIR / "events_with_stints.parquet"
df_events.to_parquet(merged_path)
print("Saved merged events:", merged_path)



=== Stints Merge ===
Loaded events: 1,658,412
Loaded stints: 2,212,064
Stints columns: ['game_id', 'period', 'period_time_start', 'period_time_end', 'game_stint', 'n_home_skaters', 'n_away_skaters', 'is_home_net_empty', 'is_away_net_empty', 'home_score', 'away_score']
Stints total rows: 187807
Stints sample:
                                 game_id  period period_time_start  \
0   00b0366a-95c6-5250-2dae-e3dd5c4198bc       1              0E-9   
12  00b0366a-95c6-5250-2dae-e3dd5c4198bc       1      31.000000000   
24  00b0366a-95c6-5250-2dae-e3dd5c4198bc       1      36.000000000   
36  00b0366a-95c6-5250-2dae-e3dd5c4198bc       1      38.000000000   
48  00b0366a-95c6-5250-2dae-e3dd5c4198bc       1      42.000000000   

   period_time_end  game_stint  n_home_skaters  n_away_skaters  \
0     31.000000000           1               5               5   
12    36.000000000           2               5               5   
24    38.000000000           3               5               5   
36  

### Establish Continuity-First Event Order

Preserve native event order from source data and derive stable ordering ids for downstream continuous-state modeling.
No strict faceoff-first tie-breaking is enforced; contextual post-whistle penalties are retained.

In [2]:
import pandas as pd
import numpy as np

# Load merged events
merged_path = OUTPUT_DIR / "events_with_stints.parquet"
df_events = pd.read_parquet(merged_path)

print(f"Preparing {len(df_events):,} events with continuity-first ordering...")

# Ensure numeric period_time for downstream time features
# (leave NaN as-is; they are informative for some boundary events)
df_events['period_time'] = pd.to_numeric(df_events['period_time'], errors='coerce')

# ---------------------------------------------------------------------------
# 1) Preserve ingest/native order by default; only corrective-sort if abnormal
#    Abnormal = period_time delta < -1 within (game_id, period, sequence_id)
# ---------------------------------------------------------------------------
df_events['_ingest_order'] = np.arange(len(df_events), dtype=np.int64)
seq_keys = ['game_id', 'period', 'sequence_id']

delta = df_events.groupby(seq_keys, sort=False)['period_time'].diff()
abnormal_mask = delta < -1
n_abnormal = int(abnormal_mask.sum())
n_abnormal_seq = int(df_events.loc[abnormal_mask, seq_keys].drop_duplicates().shape[0])

print(f"Abnormal period_time deltas (< -1): {n_abnormal:,} across {n_abnormal_seq:,} sequences")

if n_abnormal > 0:
    # Corrective temporal sort only when disorder is detected
    sort_keys = [c for c in ['game_id', 'period', 'sequence_id', 'period_time', 'sl_event_id'] if c in df_events.columns]
    df_events.sort_values(by=sort_keys, ascending=True, kind='mergesort', inplace=True)
    df_events.reset_index(drop=True, inplace=True)
    print(f"Applied corrective sort with keys: {sort_keys}")
else:
    # Keep original parquet ingest order
    sort_keys = ['_ingest_order']
    df_events.sort_values(by=sort_keys, ascending=True, kind='mergesort', inplace=True)
    df_events.reset_index(drop=True, inplace=True)
    print("No abnormality detected; preserving ingest/native order.")

# ---------------------------------------------------------------------------
# 2) Rebuild ordering ids from preserved/corrected order
# ---------------------------------------------------------------------------
df_events['sequence_event_id'] = (
    df_events.groupby(['game_id', 'sequence_id']).cumcount() + 1
)
df_events['game_event_id'] = df_events.groupby('game_id').cumcount() + 1

# ---------------------------------------------------------------------------
# 3) Explicit boundary markers for downstream continuous-state modeling
# ---------------------------------------------------------------------------
boundary_events = {'whistle', 'goal', 'end_of_period'}
df_events['is_boundary_event'] = df_events['event_type'].isin(boundary_events).astype(np.int8)
df_events['is_end_of_period'] = (df_events['event_type'] == 'end_of_period').astype(np.int8)

# ---------------------------------------------------------------------------
# 4) Add game_time_sec while preserving period context
#    - Detect per (game_id, period) whether period_time behaves as countdown
# ---------------------------------------------------------------------------
df_events['_period_elapsed_sec'] = df_events['period_time']

for (game_id, period), idx in df_events.groupby(['game_id', 'period']).groups.items():
    period_vals = df_events.loc[idx, 'period_time'].to_numpy(dtype=float)

    if len(period_vals) <= 1:
        continue

    deltas = np.diff(period_vals)
    counts_down = (deltas < 0).sum() > (deltas > 0).sum()

    if counts_down:
        df_events.loc[idx, '_period_elapsed_sec'] = 1200.0 - period_vals

period_idx = pd.to_numeric(df_events['period'], errors='coerce').fillna(1.0)
df_events['game_time_sec'] = (period_idx - 1.0) * 1200.0 + df_events['_period_elapsed_sec']

df_events.drop(columns=['_ingest_order', '_period_elapsed_sec'], inplace=True)

# Keep current order stable (do not resort by game_time_sec unless abnormality triggered above)
df_events.reset_index(drop=True, inplace=True)

# Refresh game_event_id after final stable order
df_events['game_event_id'] = df_events.groupby('game_id').cumcount() + 1

print("\n=== Ordering + Continuity Summary ===")
print(f"Primary ordering mode: {'corrective temporal sort' if n_abnormal > 0 else 'ingest/native order preserved'}")
print(f"Boundary events flagged: {df_events['is_boundary_event'].sum():,}")
print(f"End-of-period events flagged: {df_events['is_end_of_period'].sum():,}")
print(f"Rows with game_time_sec populated: {df_events['game_time_sec'].notna().sum():,} / {len(df_events):,}")

if 'event_type' in df_events.columns:
    keep_types = ['whistle', 'faceoff', 'icing', 'penalty', 'penaltydrawn', 'goal', 'end_of_period']
    type_counts = df_events['event_type'].value_counts()
    print("\nKey event-type retention check:")
    for t in keep_types:
        print(f"  {t}: {int(type_counts.get(t, 0)):,}")

print("\n✓ Continuity-first ordering complete (no unconditional game-time resort)")

Preparing 1,658,412 events with continuity-first ordering...
Abnormal period_time deltas (< -1): 0 across 0 sequences
No abnormality detected; preserving ingest/native order.

=== Ordering + Continuity Summary ===
Primary ordering mode: ingest/native order preserved
Boundary events flagged: 28,339
End-of-period events flagged: 12
Rows with game_time_sec populated: 1,658,412 / 1,658,412

Key event-type retention check:
  whistle: 25,517
  faceoff: 56,190
  icing: 4,052
  penalty: 3,833
  penaltydrawn: 3,748
  goal: 2,810
  end_of_period: 12

✓ Continuity-first ordering complete (no unconditional game-time resort)


## 3.1 Add Distance and Angle to Net for All Events

Calculate distance and angle to net for the base events file (before tracking merge).

In [3]:
import pandas as pd
import numpy as np

print(f"Loaded {len(df_events):,} events")
print(f"Columns: {df_events.columns.tolist()}")

# Calculate distance to net (right side at x=89, y=0)
df_events['distance_to_net_event'] = np.sqrt(
    (89 - df_events['x_adj'])**2 + 
    (0 - df_events['y_adj'])**2
)

# Calculate angle to net (in radians)
df_events['angle_to_net_event'] = np.arctan2(
    df_events['y_adj'], 
    89 - df_events['x_adj']
)

# Show statistics
print("\n--- Distance to Net ---")
print(df_events['distance_to_net_event'].describe())

print("\n--- Angle to Net (radians) ---")
print(df_events['angle_to_net_event'].describe())

# Show sample
print("\n--- Sample Events ---")
print(df_events[['event_type', 'x_adj', 'y_adj', 'distance_to_net_event', 'angle_to_net_event']].head(10))

# Save to parquet file
events_path = OUTPUT_DIR / "events_clean_description_stints.parquet"
df_events.to_parquet(events_path, index=False)
print(f"\n✅ Saved updated file with distance_to_net_event and angle_to_net_event to:\n{events_path}")

Loaded 1,658,412 events
Columns: ['game_id', 'period', 'period_time', 'game_stint', 'sequence_id', 'player_id', 'team_id', 'opp_team_id', 'event_type', 'outcome', 'flags', 'flags_clean', 'description', 'description_clean', 'detail', 'x', 'y', 'x_adj', 'y_adj', 'has_tracking_data', 'event_player_tracked', 'dest_x_adj', 'dest_y_adj', 'sl_event_id', 'sl_xg_all_shots', 'embedding_text_clean', 'period_time_start', 'period_time_end', 'n_home_skaters', 'n_away_skaters', 'is_home_net_empty', 'is_away_net_empty', 'home_score', 'away_score', 'score_differential_home', 'score_differential_away', 'sequence_event_id', 'game_event_id', 'is_boundary_event', 'is_end_of_period', 'game_time_sec']

--- Distance to Net ---
count    1.633158e+06
mean     1.031192e+02
std      5.398944e+01
min      1.016926e+00
25%      5.562886e+01
50%      1.032156e+02
75%      1.536949e+02
max      1.895934e+02
Name: distance_to_net_event, dtype: float64

--- Angle to Net (radians) ---
count    1.633158e+06
mean    -1.45

## 4. Tracking Data

Load tracking data and outline steps to create relative position snapshots.

In [2]:
#load events if not already loaded
events_path = OUTPUT_DIR / "events_clean_description_stints.parquet"
df_events = pd.read_parquet(events_path)

#Get list of columns
print(df_events.columns)

Index(['game_id', 'period', 'period_time', 'game_stint', 'sequence_id',
       'player_id', 'team_id', 'opp_team_id', 'event_type', 'outcome', 'flags',
       'flags_clean', 'description', 'description_clean', 'detail', 'x', 'y',
       'x_adj', 'y_adj', 'has_tracking_data', 'event_player_tracked',
       'dest_x_adj', 'dest_y_adj', 'sl_event_id', 'sl_xg_all_shots',
       'embedding_text_clean', 'period_time_start', 'period_time_end',
       'n_home_skaters', 'n_away_skaters', 'is_home_net_empty',
       'is_away_net_empty', 'home_score', 'away_score',
       'score_differential_home', 'score_differential_away',
       'sequence_event_id', 'game_event_id', 'is_boundary_event',
       'is_end_of_period', 'game_time_sec', 'distance_to_net_event',
       'angle_to_net_event'],
      dtype='object')


In [3]:
print("\n=== Tracking Merge (Event-Level Filter) ===")
    

# 1) Load tracking
tracking_path = DATA_DIR / "tracking.parquet"
if not tracking_path.exists():
    print(f"Missing tracking file: {tracking_path}")
else:
    df_tracking = pd.read_parquet(tracking_path)
    print(f"Loaded tracking: {len(df_tracking):,}")

    # 2) Inspect columns
    print("Tracking columns:", sorted(df_tracking.columns.tolist()))
    print("Events columns (pre-merge):", sorted(df_events.columns.tolist()))

    # 3) Create tracking keys for merge
    required_flag_cols = ["has_tracking_data"]
    missing_flag_cols = [c for c in required_flag_cols if c not in df_events.columns]
    if missing_flag_cols:
        raise KeyError(f"Missing tracking flag columns: {missing_flag_cols}")

    df_events["tracking_key"] = (
        df_events["game_id"].astype(str) + "_" + df_events["sl_event_id"].astype(str)
    )
    df_tracking["tracking_key"] = (
        df_tracking["game_id"].astype(str) + "_" + df_tracking["sl_event_id"].astype(str)
    )

    # 4) Filter to events with tracking data (includes ALL players, not just event player)
    events_for_tracking = df_events[
        df_events["has_tracking_data"] == 1
    ].copy()

    # 5) Inner join: keep only tracking rows for events with tracking data (game_id and sl_event_id are already part of tracking_key, so we only need to bring in tracking_key and event identifiers for validation)
    df_tracking_filtered = df_tracking.merge(
        events_for_tracking[["tracking_key", "sequence_id", "game_event_id", "sequence_event_id"]],
        on="tracking_key",
        how="inner"
    )

    print("Events with tracking data:", len(events_for_tracking))
    print("Tracking rows matched (all players):", len(df_tracking_filtered))
    print("Unique events with tracking:", df_tracking_filtered["tracking_key"].nunique())
    print("Total events (unchanged):", len(df_events))


=== Tracking Merge (Event-Level Filter) ===
Loaded tracking: 13,529,224
Tracking columns: ['game_id', 'player_id', 'player_name', 'sl_event_id', 'team_id', 'team_name', 'tracking_vel_x', 'tracking_vel_y', 'tracking_x', 'tracking_y']
Events columns (pre-merge): ['angle_to_net_event', 'away_score', 'description', 'description_clean', 'dest_x_adj', 'dest_y_adj', 'detail', 'distance_to_net_event', 'embedding_text_clean', 'event_player_tracked', 'event_type', 'flags', 'flags_clean', 'game_event_id', 'game_id', 'game_stint', 'game_time_sec', 'has_tracking_data', 'home_score', 'is_away_net_empty', 'is_boundary_event', 'is_end_of_period', 'is_home_net_empty', 'n_away_skaters', 'n_home_skaters', 'opp_team_id', 'outcome', 'period', 'period_time', 'period_time_end', 'period_time_start', 'player_id', 'score_differential_away', 'score_differential_home', 'sequence_event_id', 'sequence_id', 'sl_event_id', 'sl_xg_all_shots', 'team_id', 'x', 'x_adj', 'y', 'y_adj']
Events with tracking data: 1506750
T

### 4.1 Tracking Data: Detailed Exploration

Inspect tracking data structure, identify goalie presence, and verify coordinate system.

In [4]:
print("=" * 80)
print("TRACKING DATA: BASIC STRUCTURE")
print("=" * 80)

# Display basic info
print(f"\nTotal tracking rows: {len(df_tracking_filtered):,}")
print(f"Unique events with tracking: {df_tracking_filtered['tracking_key'].nunique():,}")
print(f"Unique games: {df_tracking_filtered['game_id'].nunique():,}")
print(f"Unique players: {df_tracking_filtered['player_id'].nunique():,}")

print("\n" + "-" * 80)
print("Column names and types:")
print("-" * 80)
print(df_tracking_filtered.dtypes)

print("\n" + "-" * 80)
print("First 20 rows (sample):")
print("-" * 80)
df_tracking_filtered.head(20).to_string()

TRACKING DATA: BASIC STRUCTURE

Total tracking rows: 12,496,599
Unique events with tracking: 1,506,603
Unique games: 479
Unique players: 771

--------------------------------------------------------------------------------
Column names and types:
--------------------------------------------------------------------------------
game_id               object
sl_event_id            int64
team_id               object
team_name             object
player_id             object
player_name           object
tracking_x           float64
tracking_y           float64
tracking_vel_x       float64
tracking_vel_y       float64
tracking_key          object
sequence_id            int64
game_event_id          int64
sequence_event_id      int64
dtype: object

--------------------------------------------------------------------------------
First 20 rows (sample):
--------------------------------------------------------------------------------


"                                 game_id  sl_event_id                               team_id team_name                             player_id           player_name  tracking_x  tracking_y  tracking_vel_x  tracking_vel_y                            tracking_key  sequence_id  game_event_id  sequence_event_id\n0   00b0366a-95c6-5250-2dae-e3dd5c4198bc            1  d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019  Monsters  0f052c34-6cd2-f4fd-a861-d9051a8e86e4          Angle, Tyler    1.735564  -12.985565             NaN             NaN  00b0366a-95c6-5250-2dae-e3dd5c4198bc_1            1              1                  1\n1   00b0366a-95c6-5250-2dae-e3dd5c4198bc            1  6cac12e2-0546-2c1a-689f-ab26d8a6355a  Griffins                                  None                  None   -2.237533   10.006562        0.688983       -6.692981  00b0366a-95c6-5250-2dae-e3dd5c4198bc_1            1              1                  1\n2   00b0366a-95c6-5250-2dae-e3dd5c4198bc            1  d7ff41e7-8310-f2ab-3c79-6a

### 4.2 Player Count Analysis & Data Cleaning

Remove None player_ids, deduplicate tracking entries, and analyze player counts per event.

In [5]:
print("=" * 80)
print("PLAYER COUNT DISTRIBUTION PER EVENT")
print("=" * 80)

# Remove player_id = None/NaN (non-player entities or tracking errors)
if 'player_id' in df_tracking_filtered.columns:
    non_player_rows = df_tracking_filtered[df_tracking_filtered['player_id'].isna() | (df_tracking_filtered['player_id'] == 'None')]
    if len(non_player_rows) > 0:
        print(f"Removing {len(non_player_rows):,} rows with non-player IDs.")
        df_tracking_filtered = df_tracking_filtered[~(df_tracking_filtered['player_id'].isna() | (df_tracking_filtered['player_id'] == 'None'))].copy()
else:
    print("Warning: 'player_id' column not found in tracking data.")

# Remove duplicate player entries per event (same player_id, event, exact same coordinates)
print("\nRemoving duplicate player entries per event...")
before_dedup = len(df_tracking_filtered)
df_tracking_filtered = df_tracking_filtered.drop_duplicates(
    subset=['tracking_key', 'player_id'],
    keep='first'
).copy()
after_dedup = len(df_tracking_filtered)
print(f"  Removed {before_dedup - after_dedup:,} duplicate entries")
print(f"  Remaining tracking rows: {after_dedup:,}")

# Count players per event
players_per_event = df_tracking_filtered.groupby('tracking_key').size()
print(f"\nStatistics:")
print(f"  Mean: {players_per_event.mean():.2f}")
print(f"  Median: {players_per_event.median():.0f}")
print(f"  Min: {players_per_event.min()}")
print(f"  Max: {players_per_event.max()}")

print("\n" + "-" * 80)
print("Distribution of player counts:")
print("-" * 80)
count_dist = players_per_event.value_counts().sort_index()
print(count_dist.to_string())

# Sample event to verify deduplication
events_with_tracking = players_per_event[players_per_event > 0].index.tolist()
if len(events_with_tracking) > 0:
    print("\n" + "-" * 80)
    print("SAMPLE: First event (after deduplication)")
    print("-" * 80)
    sample_key = events_with_tracking[0]
    sample_event = df_tracking_filtered[df_tracking_filtered['tracking_key'] == sample_key]
    
    print(f"Event: {sample_key}")
    print(f"Players tracked: {len(sample_event)}")
    
    # Display all players for this event
    display_cols = ['player_id', 'player_name', 'team_name', 'tracking_x', 'tracking_y']
    available_display = [c for c in display_cols if c in sample_event.columns]
    print("\n" + sample_event[available_display].to_string(index=False))

PLAYER COUNT DISTRIBUTION PER EVENT
Removing 3,879,947 rows with non-player IDs.

Removing duplicate player entries per event...
  Removed 654 duplicate entries
  Remaining tracking rows: 8,615,998

Statistics:
  Mean: 5.74
  Median: 6
  Min: 1
  Max: 14

--------------------------------------------------------------------------------
Distribution of player counts:
--------------------------------------------------------------------------------
1      23794
2      58196
3     117773
4     194073
5     263096
6     293150
7     262968
8     178740
9      82609
10     21627
11      3379
12       323
13        21
14         4

--------------------------------------------------------------------------------
SAMPLE: First event (after deduplication)
--------------------------------------------------------------------------------
Event: 00b0366a-95c6-5250-2dae-e3dd5c4198bc_1
Players tracked: 7

                           player_id          player_name team_name  tracking_x  tracking_y
0f052c

### Inspect Events with >10 Players Tracked

Show examples of events where more than 10 players were detected in the tracking data.

In [6]:
# Filter events with >10 players tracked
print("=" * 80)
print("EVENTS WITH MORE THAN 10 PLAYERS TRACKED")
print("=" * 80)

for player_count in [11, 12, 13, 14]:
    # Find events with exactly this many players
    events_with_n = players_per_event[players_per_event == player_count]
    
    if len(events_with_n) == 0:
        print(f"\n{player_count} players: No events found")
        continue
    
    print(f"\n{'-'*80}")
    print(f"{player_count} PLAYERS - Found {len(events_with_n)} events (showing first 3)")
    print(f"{'-'*80}")
    
    # Show up to 3 examples
    for idx, (tracking_key, count) in enumerate(events_with_n.head(3).items()):
        sample = df_tracking_filtered[df_tracking_filtered['tracking_key'] == tracking_key].copy()
        
        print(f"\nExample {idx+1}: Event {tracking_key}")
        print(f"  Players tracked: {len(sample)}")
        
        # Get event details from the tracking key
        game_id = sample['game_id'].iloc[0]
        game_event_id = sample['game_event_id'].iloc[0]
        
        # Try to get event type and description from events if available
        if 'event_type' in sample.columns:
            event_type = sample['event_type'].iloc[0] if pd.notna(sample['event_type'].iloc[0]) else 'unknown'
            print(f"  Event type: {event_type}")
        
        # Show player details
        display_cols = ['player_name', 'team_name', 'tracking_x', 'tracking_y']
        available_display = [col for col in display_cols if col in sample.columns]
        
        if available_display:
            print(f"\n  Players:")
            for i, row in sample[available_display].iterrows():
                player_info = " | ".join([f"{col}: {row[col]}" for col in available_display])
                print(f"    {player_info}")

print(f"\n{'='*80}")
print("INSPECTION COMPLETE")
print("Note: Expected player structure is 1 actor + 5 teammates + 6 opponents = 12 players")
print("Events with >12 or <12 may indicate tracking anomalies or unusual game situations")
print("=" * 80)

EVENTS WITH MORE THAN 10 PLAYERS TRACKED

--------------------------------------------------------------------------------
11 PLAYERS - Found 3379 events (showing first 3)
--------------------------------------------------------------------------------

Example 1: Event 021989d7-459d-f1e0-c308-aab61c8b0037_2443
  Players tracked: 11

  Players:
    player_name: Farinacci, John | team_name: Bruins | tracking_x: 29.02887232 | tracking_y: 41.7814974
    player_name: Koppanen, Joona | team_name: Penguins | tracking_x: -6.607611759999999 | tracking_y: 10.990814
    player_name: Regula, Alec | team_name: Bruins | tracking_x: -30.331365799999997 | tracking_y: 1.1679790399999999
    player_name: Poulin, Samuel | team_name: Penguins | tracking_x: 3.64501324 | tracking_y: -34.76378064
    player_name: Kuntar, Trevor | team_name: Bruins | tracking_x: 7.7657482799999995 | tracking_y: 30.29527656
    player_name: Puljujarvi, Jesse | team_name: Penguins | tracking_x: 9.183071159999999 | tracking_y: 

### 4.3 Filter Out Goalies from Tracking

Load players metadata, identify goalies, and remove them from tracking data. Goalies excluded due to low coverage (4.5% overall, 7.9% of shots).

In [7]:
print("=" * 80)
print("FILTER OUT GOALIES FROM TRACKING DATA")
print("=" * 80)

# 1. Load players metadata
players_path = DATA_DIR / "players.parquet"
if not players_path.exists():
    print(f"⚠ Players file not found: {players_path}")
    print("Skipping goalie filtering - proceeding with all players")
else:
    df_players = pd.read_parquet(players_path)
    print(f"Loaded players: {len(df_players):,}")
    
    # 2. Identify goalie IDs from position column
    position_cols = [col for col in df_players.columns if 'position' in col.lower()]
    
    if position_cols:
        pos_col = position_cols[0]
        print(f"Position column found: {pos_col}")
        
        # Identify goalies (position contains 'G' or 'GOAL')
        df_players['is_goalie'] = df_players[pos_col].str.upper().str.contains('G|GOAL', na=False, regex=True)
        goalie_ids = df_players[df_players['is_goalie']]['player_id'].unique()
        
        print(f"Goalies identified: {len(goalie_ids):,}")
        print(f"Sample goalies: {df_players[df_players['is_goalie']][['player_name', pos_col]].head(3).to_string(index=False)}")
        
        # 3. Filter out goalies from tracking data
        print(f"\nTracking data before filtering: {len(df_tracking_filtered):,} rows")
        
        goalies_in_tracking = df_tracking_filtered['player_id'].isin(goalie_ids).sum()
        print(f"Goalie tracking rows to remove: {goalies_in_tracking:,}")
        
        df_tracking_filtered = df_tracking_filtered[~df_tracking_filtered['player_id'].isin(goalie_ids)].copy()
        print(f"Tracking data after filtering: {len(df_tracking_filtered):,} rows")
        
    else:
        print("\n⚠ No position column found in players data")
        print("Skipping goalie filtering - proceeding with all players")

# 4. Save filtered tracking data
output_tracking_path = OUTPUT_DIR / "tracking_filtered.parquet"
df_tracking_filtered.to_parquet(output_tracking_path, index=False)
print(f"\n✓ Saved filtered tracking data to: {output_tracking_path}")
print(f"  Events with tracking: {df_tracking_filtered['tracking_key'].nunique():,}")
print(f"  Total tracking rows: {len(df_tracking_filtered):,}")
print(f"  Unique players (skaters only): {df_tracking_filtered['player_id'].nunique():,}")

FILTER OUT GOALIES FROM TRACKING DATA
Loaded players: 1,172
Position column found: position_group
Goalies identified: 114
Sample goalies:     player_name position_group
  Garand, Dylan              G
    Ollas, Hugo              G
Brodeur, Jeremy              G

Tracking data before filtering: 8,615,998 rows
Goalie tracking rows to remove: 68,126
Tracking data after filtering: 8,547,872 rows

✓ Saved filtered tracking data to: x:\My Files\Python\Sports Analytics Projects\Hockey\HALO\Tensor-Ready Data\tracking_filtered.parquet
  Events with tracking: 1,499,415
  Total tracking rows: 8,547,872
  Unique players (skaters only): 685


### 4.4 Coordinate System Verification

Check if tracking_x/tracking_y are raw coordinates or already adjusted. Compare event player's tracking position to event x/y coordinates.

In [8]:
print("=" * 80)
print("COORDINATE SYSTEM VERIFICATION")
print("=" * 80)

# Load df_tracking_filtered if not in locals
if 'df_tracking_filtered' not in locals():
    tracking_filtered_path = OUTPUT_DIR / "tracking_filtered.parquet"
    if tracking_filtered_path.exists():
        df_tracking_filtered = pd.read_parquet(tracking_filtered_path)
        print(f"Loaded tracking_filtered from {tracking_filtered_path}")
        print(f"  Rows: {len(df_tracking_filtered):,}")
        print(f"  Events: {df_tracking_filtered['tracking_key'].nunique():,}")
    else:
        raise FileNotFoundError(f"tracking_filtered.parquet not found at {tracking_filtered_path}")
else:
    print("df_tracking_filtered already in memory")

# Merge tracking with full event data to get event coordinates
df_tracking_with_coords = df_tracking_filtered.merge(
    df_events[['game_id', 'sl_event_id', 'player_id', 'team_id', 'x', 'y', 'x_adj', 'y_adj']],
    on=['game_id', 'sl_event_id'],
    how='left',
    suffixes=('_tracking', '_event')
)

print(f"Merged tracking with event coords: {len(df_tracking_with_coords):,} rows")

# For events where the event player is tracked, compare their tracking position to event position
event_player_tracked = df_tracking_with_coords[
    df_tracking_with_coords['player_id_tracking'] == df_tracking_with_coords['player_id_event']
].copy()

print(f"\nEvent players in tracking data: {len(event_player_tracked):,}")

if len(event_player_tracked) > 0:
    # Calculate differences
    event_player_tracked['diff_x_raw'] = abs(event_player_tracked['tracking_x'] - event_player_tracked['x'])
    event_player_tracked['diff_y_raw'] = abs(event_player_tracked['tracking_y'] - event_player_tracked['y'])
    event_player_tracked['diff_x_adj'] = abs(event_player_tracked['tracking_x'] - event_player_tracked['x_adj'])
    event_player_tracked['diff_y_adj'] = abs(event_player_tracked['tracking_y'] - event_player_tracked['y_adj'])
    
    print("\n" + "-" * 80)
    print("Comparison: tracking coords vs event coords")
    print("-" * 80)
    print(f"\nMean absolute difference (tracking_x vs x):     {event_player_tracked['diff_x_raw'].mean():.4f}")
    print(f"Mean absolute difference (tracking_y vs y):     {event_player_tracked['diff_y_raw'].mean():.4f}")
    print(f"Mean absolute difference (tracking_x vs x_adj): {event_player_tracked['diff_x_adj'].mean():.4f}")
    print(f"Mean absolute difference (tracking_y vs y_adj): {event_player_tracked['diff_y_adj'].mean():.4f}")
    
    # Determine which coordinate system tracking uses
    raw_diff = event_player_tracked['diff_x_raw'].mean() + event_player_tracked['diff_y_raw'].mean()
    adj_diff = event_player_tracked['diff_x_adj'].mean() + event_player_tracked['diff_y_adj'].mean()
    
    print("\n" + "-" * 80)
    print("CONCLUSION:")
    print("-" * 80)
    if raw_diff < adj_diff:
        print("✓ Tracking coordinates match RAW event coordinates (x, y)")
        print("  → We need to apply flip transformation to align with x_adj, y_adj")
        tracking_coord_system = 'raw'
    else:
        print("✓ Tracking coordinates match ADJUSTED event coordinates (x_adj, y_adj)")
        print("  → No transformation needed")
        tracking_coord_system = 'adjusted'
        
    # Show sample comparison
    print("\n" + "-" * 80)
    print("Sample comparison (first 5 event players):")
    print("-" * 80)
    sample_cols = ['tracking_x', 'tracking_y', 'x', 'y', 'x_adj', 'y_adj', 
                   'diff_x_raw', 'diff_y_raw', 'diff_x_adj', 'diff_y_adj']
    available_sample = [c for c in sample_cols if c in event_player_tracked.columns]
    print(event_player_tracked[available_sample].head(5).to_string(index=False))
else:
    print("\n⚠ No event players found in tracking data - cannot verify coordinate system")
    tracking_coord_system = 'raw'
    print("  → Assuming tracking uses RAW coordinates (x, y)")

print(f"\n✓ Tracking coordinate system: {tracking_coord_system.upper()}")

COORDINATE SYSTEM VERIFICATION
df_tracking_filtered already in memory
Merged tracking with event coords: 8,548,514 rows

Event players in tracking data: 1,030,916

--------------------------------------------------------------------------------
Comparison: tracking coords vs event coords
--------------------------------------------------------------------------------

Mean absolute difference (tracking_x vs x):     3.6033
Mean absolute difference (tracking_y vs y):     3.3537
Mean absolute difference (tracking_x vs x_adj): 52.8188
Mean absolute difference (tracking_y vs y_adj): 24.8724

--------------------------------------------------------------------------------
CONCLUSION:
--------------------------------------------------------------------------------
✓ Tracking coordinates match RAW event coordinates (x, y)
  → We need to apply flip transformation to align with x_adj, y_adj

--------------------------------------------------------------------------------
Sample comparison (first

### 4.5 Tracking Transformation Pipeline: Event Context Merge

Merge tracking data with event information to get acting team, coordinates, and flip logic.

In [9]:
print("=" * 80)
print("STEP 1: MERGE TRACKING WITH EVENT CONTEXT")
print("=" * 80)

# Prepare event subset with necessary columns
# Note: sequence_id, game_event_id, sequence_event_id already in df_tracking_filtered from cell 4.2
# Rename team_id to acting_team_id BEFORE merge to avoid suffix confusion
events_subset = df_events[[
    'game_id', 'sl_event_id', 'team_id', 'x', 'y', 'x_adj', 'y_adj',
    'event_type', 'is_boundary_event', 'is_end_of_period'
]].copy()

# Rename team_id to acting_team_id for clarity (this is the team performing the event)
events_subset = events_subset.rename(columns={'team_id': 'acting_team_id'})

# Create flip_required column
# True if coordinates were adjusted (x != x_adj OR y != y_adj), accounting for NaN
events_subset['flip_required'] = (
    (events_subset['x'] != events_subset['x_adj']) |
    (events_subset['y'] != events_subset['y_adj'])
)

# Handle NaN cases: if both x and x_adj are NaN, set flip_required = False
both_nan_x = events_subset['x'].isna() & events_subset['x_adj'].isna()
both_nan_y = events_subset['y'].isna() & events_subset['y_adj'].isna()
events_subset.loc[both_nan_x & both_nan_y, 'flip_required'] = False

print(f"Events prepared: {len(events_subset):,}")
print(f"Events requiring coordinate flip: {events_subset['flip_required'].sum():,}")
print(f"Events NOT requiring flip: {(~events_subset['flip_required']).sum():,}")
print(f"Events with NaN coordinates: {events_subset['x_adj'].isna().sum():,}")

print(f"\nColumn check before merge:")
print(f"  df_tracking_filtered columns: {sorted(df_tracking_filtered.columns.tolist())}")
print(f"  events_subset columns: {sorted(events_subset.columns.tolist())}")

# Merge tracking with event context
# No suffixes needed - no overlapping non-key columns
df_tracking_enriched = df_tracking_filtered.merge(
    events_subset,
    on=['game_id', 'sl_event_id'],
    how='left'
)

print(f"\n✓ Tracking enriched with event context: {len(df_tracking_enriched):,} rows")
print(f"  Columns after merge: {sorted(df_tracking_enriched.columns.tolist())}")

# Verify merge
if 'acting_team_id' in df_tracking_enriched.columns:
    missing_context = df_tracking_enriched['acting_team_id'].isna().sum()
    if missing_context > 0:
        missing_events = df_tracking_enriched[df_tracking_enriched['acting_team_id'].isna()]['tracking_key'].nunique()
        print(f"\n✓ Tracking enriched successfully")
        print(f"  Note: {missing_context:,} tracking rows ({missing_events:,} events) have no acting team")
        print(f"  These rows are retained as boundary/system context for continuous-state modeling")
    else:
        print(f"\n✓ All tracking rows successfully merged with event context")
else:
    print(f"\n⚠ Warning: Could not find acting_team_id column after merge")
    print(f"  Available columns: {[c for c in df_tracking_enriched.columns if 'team' in c.lower()]}")

STEP 1: MERGE TRACKING WITH EVENT CONTEXT
Events prepared: 1,658,412
Events requiring coordinate flip: 815,254
Events NOT requiring flip: 843,158
Events with NaN coordinates: 25,254

Column check before merge:
  df_tracking_filtered columns: ['game_event_id', 'game_id', 'player_id', 'player_name', 'sequence_event_id', 'sequence_id', 'sl_event_id', 'team_id', 'team_name', 'tracking_key', 'tracking_vel_x', 'tracking_vel_y', 'tracking_x', 'tracking_y']
  events_subset columns: ['acting_team_id', 'event_type', 'flip_required', 'game_id', 'is_boundary_event', 'is_end_of_period', 'sl_event_id', 'x', 'x_adj', 'y', 'y_adj']

✓ Tracking enriched with event context: 8,548,514 rows
  Columns after merge: ['acting_team_id', 'event_type', 'flip_required', 'game_event_id', 'game_id', 'is_boundary_event', 'is_end_of_period', 'player_id', 'player_name', 'sequence_event_id', 'sequence_id', 'sl_event_id', 'team_id', 'team_name', 'tracking_key', 'tracking_vel_x', 'tracking_vel_y', 'tracking_x', 'tracking

In [10]:
print("=" * 80)
print("EVENT TYPE ANALYSIS: Missing team_id")
print("=" * 80)

# Check what event types have team_id = None in df_events
events_no_team = df_events[df_events['team_id'].isna()].copy()
print(f"\nEvents with team_id = None: {len(events_no_team):,} ({len(events_no_team)/len(df_events)*100:.2f}%)")

if len(events_no_team) > 0:
    print(f"\nEvent type breakdown (team_id = None):")
    event_type_counts = events_no_team['event_type'].value_counts()
    for event_type, count in event_type_counts.items():
        pct = count / len(events_no_team) * 100
        print(f"  {event_type}: {count:,} ({pct:.1f}%)")

    # Check if these are tracked
    events_no_team['tracking_key'] = events_no_team['game_id'].astype(str) + "_" + events_no_team['sl_event_id'].astype(str)
    tracked_no_team = events_no_team[events_no_team['has_tracking_data'] == 1]
    print(f"\nEvents with team_id=None AND has_tracking_data=1: {len(tracked_no_team):,}")

    if len(tracked_no_team) > 0:
        print(f"\nTracked event types (team_id = None):")
        tracked_types = tracked_no_team['event_type'].value_counts()
        for event_type, count in tracked_types.items():
            print(f"  {event_type}: {count:,}")

    # Count tracking rows for these events
    tracked_keys_no_team = set(tracked_no_team['tracking_key'])
    tracking_rows_no_team = df_tracking_enriched[df_tracking_enriched['tracking_key'].isin(tracked_keys_no_team)]
    print(f"\nTracking rows for team_id=None events: {len(tracking_rows_no_team):,}")
    print(f"Expected missing context rows: {len(tracking_rows_no_team):,}")
    print(f"Actual missing context rows: {missing_context:,}")
    print(f"Match: {'✓ YES' if abs(len(tracking_rows_no_team) - missing_context) < 100 else '✗ NO'}")

print("\n" + "=" * 80)
print("CONCLUSION:")
print("=" * 80)
print("Rows with missing team_id are retained as valid boundary/system-context events.")
print("If ranking context players is not possible for those rows, tracking features are zero-padded later.")
print("This preserves continuity for downstream rolling-window supervision.")

EVENT TYPE ANALYSIS: Missing team_id

Events with team_id = None: 25,254 (1.52%)

Event type breakdown (team_id = None):
  whistle: 25,254 (100.0%)

Events with team_id=None AND has_tracking_data=1: 22,918

Tracked event types (team_id = None):
  whistle: 22,918

Tracking rows for team_id=None events: 119,189
Expected missing context rows: 119,189
Actual missing context rows: 119,189
Match: ✓ YES

CONCLUSION:
Rows with missing team_id are retained as valid boundary/system-context events.
If ranking context players is not possible for those rows, tracking features are zero-padded later.
This preserves continuity for downstream rolling-window supervision.


### 4.6 Coordinate Transformation: Standardize to Adjusted System

Apply flip transformation to align tracking coordinates with adjusted event coordinates (attacking right).

In [11]:
print("=" * 80)
print("STEP 2: COORDINATE TRANSFORMATION (STANDARDIZATION)")
print("=" * 80)

# Apply flip transformation based on flip_required flag
# If tracking coordinates are RAW and event required flip, we flip tracking coords
if tracking_coord_system == 'raw':
    print("Applying coordinate flip transformation where required...")
    
    df_tracking_enriched['x_std'] = np.where(
        df_tracking_enriched['flip_required'],
        -df_tracking_enriched['tracking_x'],
        df_tracking_enriched['tracking_x']
    )
    df_tracking_enriched['y_std'] = np.where(
        df_tracking_enriched['flip_required'],
        -df_tracking_enriched['tracking_y'],
        df_tracking_enriched['tracking_y']
    )
    
    # Also flip velocities if available
    if 'tracking_vel_x' in df_tracking_enriched.columns:
        df_tracking_enriched['vx_std'] = np.where(
            df_tracking_enriched['flip_required'],
            -df_tracking_enriched['tracking_vel_x'],
            df_tracking_enriched['tracking_vel_x']
        )
        df_tracking_enriched['vy_std'] = np.where(
            df_tracking_enriched['flip_required'],
            -df_tracking_enriched['tracking_vel_y'],
            df_tracking_enriched['tracking_vel_y']
        )
    else:
        print("  No velocity columns found in tracking data")
        
else:
    # Tracking already adjusted, just copy
    print("Tracking coords already adjusted, copying without transformation...")
    df_tracking_enriched['x_std'] = df_tracking_enriched['tracking_x']
    df_tracking_enriched['y_std'] = df_tracking_enriched['tracking_y']
    
    if 'tracking_vel_x' in df_tracking_enriched.columns:
        df_tracking_enriched['vx_std'] = df_tracking_enriched['tracking_vel_x']
        df_tracking_enriched['vy_std'] = df_tracking_enriched['tracking_vel_y']

print(f"✓ Standardized coordinates created")
print(f"  Sample standardized coords (first 5):")
coord_cols = ['tracking_x', 'tracking_y', 'x_std', 'y_std', 'flip_required']
available_coord_cols = [c for c in coord_cols if c in df_tracking_enriched.columns]
print(df_tracking_enriched[available_coord_cols].head().to_string(index=False))

# Verify: standardized coords should now align with x_adj/y_adj for event players
# (We'll check this in the next validation cell)

STEP 2: COORDINATE TRANSFORMATION (STANDARDIZATION)
Applying coordinate flip transformation where required...
✓ Standardized coordinates created
  Sample standardized coords (first 5):
 tracking_x  tracking_y     x_std      y_std  flip_required
   1.735564  -12.985565  1.735564 -12.985565          False
  18.441602   17.211287 18.441602  17.211287          False
  -2.362205    0.695538 -2.362205   0.695538          False
   1.099081   -0.456037  1.099081  -0.456037          False
  -2.552494   13.848426 -2.552494  13.848426          False


### 4.7 Role Identification & Distance Calculation

Identify teammates vs opponents, calculate relative positions and distance to event.

In [12]:
print("=" * 80)
print("STEP 3: ROLE IDENTIFICATION & DISTANCE CALCULATION")
print("=" * 80)

# Identify the actor (event player) first
# Note: event player_id comes from events, tracking player_id comes from tracking
# We need to merge event player_id to identify the actor
df_tracking_enriched = df_tracking_enriched.merge(
    df_events[['game_id', 'sl_event_id', 'player_id']].rename(columns={'player_id': 'event_player_id'}),
    on=['game_id', 'sl_event_id'],
    how='left'
)

# Identify roles
df_tracking_enriched['is_actor'] = (
    df_tracking_enriched['player_id'] == df_tracking_enriched['event_player_id']
)
df_tracking_enriched['is_teammate'] = (
    (df_tracking_enriched['team_id'] == df_tracking_enriched['acting_team_id']) &
    (~df_tracking_enriched['is_actor'])  # Exclude actor from teammates
)
df_tracking_enriched['is_opponent'] = (
    (df_tracking_enriched['team_id'] != df_tracking_enriched['acting_team_id']) &
    (df_tracking_enriched['team_id'].notna())
)

print(f"\nRole distribution:")
print(f"  Actors (event players): {df_tracking_enriched['is_actor'].sum():,}")
print(f"  Teammates (excluding actor): {df_tracking_enriched['is_teammate'].sum():,}")
print(f"  Opponents: {df_tracking_enriched['is_opponent'].sum():,}")
print(f"  Unknown/Missing: {(~df_tracking_enriched['is_actor'] & ~df_tracking_enriched['is_teammate'] & ~df_tracking_enriched['is_opponent']).sum():,}")

# Calculate relative positions (centered on event location)
# rel_x, rel_y represent player position relative to puck/event location
df_tracking_enriched['rel_x'] = df_tracking_enriched['x_std'] - df_tracking_enriched['x_adj']
df_tracking_enriched['rel_y'] = df_tracking_enriched['y_std'] - df_tracking_enriched['y_adj']

# Calculate Euclidean distance from event location
df_tracking_enriched['distance'] = np.sqrt(
    df_tracking_enriched['rel_x']**2 + df_tracking_enriched['rel_y']**2
)

print(f"\nDistance statistics:")
print(f"  Mean: {df_tracking_enriched['distance'].mean():.2f}")
print(f"  Median: {df_tracking_enriched['distance'].median():.2f}")
print(f"  Min: {df_tracking_enriched['distance'].min():.2f}")
print(f"  Max: {df_tracking_enriched['distance'].max():.2f}")

print(f"\nSample with roles and distances (first 10):")
sample_cols = ['player_name', 'team_name', 'is_actor', 'is_teammate', 'is_opponent', 
               'x_std', 'y_std', 'rel_x', 'rel_y', 'distance']
available_sample = [c for c in sample_cols if c in df_tracking_enriched.columns]
print(df_tracking_enriched[available_sample].head(10).to_string(index=False))

print(f"  Total players tracked: {len(df_tracking_enriched):,}")
print(f"\n✓ Role identification and distance calculation complete")

STEP 3: ROLE IDENTIFICATION & DISTANCE CALCULATION

Role distribution:
  Actors (event players): 1,031,044
  Teammates (excluding actor): 3,346,378
  Opponents: 4,172,376
  Unknown/Missing: 0

Distance statistics:
  Mean: 27.19
  Median: 24.97
  Min: 0.00
  Max: 199.64

Sample with roles and distances (first 10):
         player_name team_name  is_actor  is_teammate  is_opponent      x_std      y_std      rel_x      rel_y  distance
        Angle, Tyler  Monsters     False        False         True   1.735564 -12.985565   1.936996 -12.230014 12.382455
      Jiricek, David  Monsters     False        False         True  18.441602  17.211287  18.643033  17.966837 25.891503
   L'Esperance, Joel  Griffins      True        False        False  -2.362205   0.695538  -2.160774   1.451088  2.602806
      McKown, Hunter  Monsters     False        False         True   1.099081  -0.456037   1.300513   0.299514  1.334557
Aston-Reese, Zachary  Griffins     False         True        False  -2.552494  1

### 4.8 Player Sorting & Ranking by Proximity

Sort players within each event by distance to event location. Rank teammates and opponents separately.

In [13]:
print("=" * 80)
print("STEP 4: SORTING & RANKING PLAYERS BY DISTANCE")
print("=" * 80)

# Ranking logic requires an acting team; boundary/system rows are kept and zero-padded later
valid_events = df_tracking_enriched['acting_team_id'].notna()
df_tracking_valid = df_tracking_enriched[valid_events].copy()

print(f"Total tracking rows: {len(df_tracking_enriched):,}")
print(f"Rows with valid acting_team_id: {len(df_tracking_valid):,}")
print(f"Rows deferred to zero-padding path: {(~valid_events).sum():,}")

# Separate actors from context players
df_actors = df_tracking_valid[df_tracking_valid['is_actor']].copy()
df_teammates = df_tracking_valid[df_tracking_valid['is_teammate']].copy()
df_opponents = df_tracking_valid[df_tracking_valid['is_opponent']].copy()

print(f"\nActors extracted: {len(df_actors):,}")
print(f"  Events with tracked actor: {df_actors['tracking_key'].nunique():,}")

# Rank teammates by distance within each event (rank 1-5, not 1-6)
df_teammates['tm_rank'] = df_teammates.groupby('tracking_key')['distance'].rank(method='first', ascending=True).astype(int)

print(f"\nTeammates ranked: {len(df_teammates):,}")
print(f"  Max teammates per event: {df_teammates.groupby('tracking_key')['tm_rank'].max().max()}")
print(f"  Mean teammates per event: {df_teammates.groupby('tracking_key')['tm_rank'].max().mean():.2f}")

# Rank opponents by distance within each event (rank 1-6)
df_opponents['opp_rank'] = df_opponents.groupby('tracking_key')['distance'].rank(method='first', ascending=True).astype(int)

print(f"\nOpponents ranked: {len(df_opponents):,}")
print(f"  Max opponents per event: {df_opponents.groupby('tracking_key')['opp_rank'].max().max()}")
print(f"  Mean opponents per event: {df_opponents.groupby('tracking_key')['opp_rank'].max().mean():.2f}")

# Quick sanity checks
print(f"\nSanity checks:")
print(f"  Unique events in valid set: {df_tracking_valid['tracking_key'].nunique():,}")
print(f"  Events with actor: {df_actors['tracking_key'].nunique():,}")
print(f"  Events with teammates: {df_teammates['tracking_key'].nunique():,}")
print(f"  Events with opponents: {df_opponents['tracking_key'].nunique():,}")

# Check for rank overflow
tm_overflow = (df_teammates['tm_rank'] > 5).sum()
opp_overflow = (df_opponents['opp_rank'] > 6).sum()
print(f"\nRank overflow check:")
print(f"  Teammates rank > 5: {tm_overflow:,}")
print(f"  Opponents rank > 6: {opp_overflow:,}")

print(f"\n✓ Player sorting and ranking complete")

STEP 4: SORTING & RANKING PLAYERS BY DISTANCE
Total tracking rows: 8,549,798
Rows with valid acting_team_id: 8,430,609
Rows deferred to zero-padding path: 119,189

Actors extracted: 1,031,044
  Events with tracked actor: 1,030,852

Teammates ranked: 3,346,378
  Max teammates per event: 20
  Mean teammates per event: 2.41

Opponents ranked: 4,053,187
  Max opponents per event: 20
  Mean opponents per event: 2.88

Sanity checks:
  Unique events in valid set: 1,476,817
  Events with actor: 1,030,852
  Events with teammates: 1,391,325
  Events with opponents: 1,407,197

Rank overflow check:
  Teammates rank > 5: 1,417
  Opponents rank > 6: 909

✓ Player sorting and ranking complete


### 4.9 Pivot to Wide Format: Tensor-Ready with Padding Masks

Transform from long format (multiple rows per event) to wide format (one row per event with fixed-size feature vectors).

**Feature Vector Structure:**
- Teammates (slots 1-6): `tm_1_x, tm_1_y, tm_1_vx, tm_1_vy, tm_1_dist, tm_1_mask, ...`
- Opponents (slots 1-6): `opp_1_x, opp_1_y, opp_1_vx, opp_1_vy, opp_1_dist, opp_1_mask, ...`

**Padding Strategy:**
- Missing positions filled with 0.0
- Mask = 1 (player present), 0 (padded slot)

In [14]:
print("=" * 80)
print("STEP 5: PIVOT TO WIDE FORMAT WITH PADDING MASKS (ACTOR-CENTRIC)")
print("=" * 80)

# Configuration: 1 Actor + 5 Teammates + 6 Opponents = 12 players
MAX_TEAMMATES = 5  # Changed from 6 to 5
MAX_OPPONENTS = 6
INCLUDE_VELOCITY = 'vx_std' in df_teammates.columns

print(f"Configuration:")
print(f"  Actor: 1 (event player)")
print(f"  Max teammates: {MAX_TEAMMATES}")
print(f"  Max opponents: {MAX_OPPONENTS}")
print(f"  Total players: {1 + MAX_TEAMMATES + MAX_OPPONENTS}")
print(f"  Include velocity: {INCLUDE_VELOCITY}")

# Prepare feature columns to pivot
feature_cols = ['rel_x', 'rel_y', 'distance']
if INCLUDE_VELOCITY:
    feature_cols.extend(['vx_std', 'vy_std'])

# First: Pivot Actor (special handling for missing actors)
print("\nPivoting actor (event player)...")

# Get all valid events
all_valid_events = df_tracking_valid[['tracking_key', 'game_id', 'sl_event_id']].drop_duplicates()

# Prepare actor features (only x, y, vel_x, vel_y - no distance since it's ~0)
actor_features = ['rel_x', 'rel_y']
if INCLUDE_VELOCITY:
    actor_features.extend(['vx_std', 'vy_std'])

# Select only needed columns from df_actors
actor_cols = ['tracking_key'] + actor_features
df_actor_pivot = df_actors[actor_cols].copy()

# Rename columns to actor_* format
rename_dict = {feat: f'actor_{feat}' for feat in actor_features}
df_actor_pivot.rename(columns=rename_dict, inplace=True)

# Add actor_mask column (1.0 where actor is tracked)
df_actor_pivot['actor_mask'] = 1.0

# Left merge with all valid events (vectorized operation)
df_actor_wide = all_valid_events[['tracking_key']].merge(
    df_actor_pivot, 
    on='tracking_key', 
    how='left'
)

# Fill missing values with 0.0 (actors not tracked)
actor_feature_cols = [f'actor_{feat}' for feat in actor_features] + ['actor_mask']
df_actor_wide[actor_feature_cols] = df_actor_wide[actor_feature_cols].fillna(0.0)

print(f"  Events with actor: {len(df_actor_wide):,}")
print(f"  Actors tracked: {df_actor_wide['actor_mask'].sum():.0f}")
print(f"  Actors missing (using event coords as fallback): {(df_actor_wide['actor_mask'] == 0).sum():.0f}")
print(f"  Columns created: {len([c for c in df_actor_wide.columns if c != 'tracking_key'])}")

# Function to pivot context players (teammates or opponents)
def pivot_role_to_wide(df_role, role_prefix, rank_col, max_slots):
    """
    Pivot role data from long to wide format with padding masks.
    
    Args:
        df_role: DataFrame with ranked players
        role_prefix: 'tm' or 'opp'
        rank_col: 'tm_rank' or 'opp_rank'
        max_slots: Maximum number of slots to create
    
    Returns:
        DataFrame with one row per event, wide format columns
    """
    # Filter to top N slots
    df_filtered = df_role[df_role[rank_col] <= max_slots].copy()
    
    # For each slot, pivot the features
    pivot_dfs = []
    
    for slot in range(1, max_slots + 1):
        slot_data = df_filtered[df_filtered[rank_col] == slot].copy()
        
        # Select columns to pivot
        cols_to_keep = ['tracking_key'] + feature_cols
        slot_pivot = slot_data[cols_to_keep].copy()
        
        # Rename columns with slot number
        rename_dict = {feat: f"{role_prefix}_{slot}_{feat}" for feat in feature_cols}
        slot_pivot.rename(columns=rename_dict, inplace=True)
        
        # Add mask column
        slot_pivot[f"{role_prefix}_{slot}_mask"] = 1.0
        
        pivot_dfs.append(slot_pivot)
    
    # Merge all slots together
    result = pivot_dfs[0]
    for df in pivot_dfs[1:]:
        result = result.merge(df, on='tracking_key', how='outer')
    
    # Fill NaN with 0.0 (padded slots)
    feature_cols_wide = [c for c in result.columns if c != 'tracking_key']
    result[feature_cols_wide] = result[feature_cols_wide].fillna(0.0)
    
    return result

# Pivot teammates (5 slots)
print("\nPivoting teammates (5 slots)...")
df_teammates_wide = pivot_role_to_wide(df_teammates, 'tm', 'tm_rank', MAX_TEAMMATES)
print(f"  Events with teammate tracking: {len(df_teammates_wide):,}")
print(f"  Columns created: {len([c for c in df_teammates_wide.columns if c != 'tracking_key'])}")

# Pivot opponents (6 slots)
print("\nPivoting opponents (6 slots)...")
df_opponents_wide = pivot_role_to_wide(df_opponents, 'opp', 'opp_rank', MAX_OPPONENTS)
print(f"  Events with opponent tracking: {len(df_opponents_wide):,}")
print(f"  Columns created: {len([c for c in df_opponents_wide.columns if c != 'tracking_key'])}")

# Merge actor + teammates + opponents
print("\nMerging actor + teammates + opponents...")
df_tracking_wide = df_actor_wide.merge(df_teammates_wide, on='tracking_key', how='left')
df_tracking_wide = df_tracking_wide.merge(df_opponents_wide, on='tracking_key', how='left')

# Fill NaN with 0.0 (represents missing data)
feature_columns = [c for c in df_tracking_wide.columns if c != 'tracking_key']
df_tracking_wide[feature_columns] = df_tracking_wide[feature_columns].fillna(0.0)

print(f"✓ Wide format tracking features: {len(df_tracking_wide):,} events")
print(f"  Total feature columns: {len(feature_columns)}")

# Summary statistics
print("\n" + "-" * 80)
print("Feature Vector Summary:")
print("-" * 80)

actor_mask_col = 'actor_mask'
teammate_mask_cols = [c for c in df_tracking_wide.columns if 'tm_' in c and '_mask' in c]
opponent_mask_cols = [c for c in df_tracking_wide.columns if 'opp_' in c and '_mask' in c]

avg_actor = df_tracking_wide[actor_mask_col].mean()
avg_teammates = df_tracking_wide[teammate_mask_cols].sum(axis=1).mean()
avg_opponents = df_tracking_wide[opponent_mask_cols].sum(axis=1).mean()

print(f"  Actor (event player) tracked: {avg_actor*100:.1f}% of events")
print(f"  Average teammates tracked per event: {avg_teammates:.2f}")
print(f"  Average opponents tracked per event: {avg_opponents:.2f}")
print(f"  Average total context players: {avg_teammates + avg_opponents:.2f}")
print(f"  Total dimensions per event: {len(feature_columns)}")

print("\n" + "-" * 80)
print("Sample feature vector (first event):")
print("-" * 80)
print(df_tracking_wide.head(1).T)

# ============================================================================
# POST-PROCESSING: Fill Missing Actor Coordinates
# ============================================================================
print("\n" + "=" * 80)
print("ACTOR IMPUTATION: Fill Missing Actor Coordinates")
print("=" * 80)

# Identify events with missing actors (actor_mask = 0)
missing_actor = df_tracking_wide['actor_mask'] == 0
num_missing = missing_actor.sum()
num_tracked = (df_tracking_wide['actor_mask'] == 1).sum()

print(f"\nActor Coverage Before Imputation:")
print(f"  Tracked (Original):  {num_tracked:,} ({num_tracked/len(df_tracking_wide)*100:.1f}%)")
print(f"  Missing (No Data):   {num_missing:,} ({num_missing/len(df_tracking_wide)*100:.1f}%)")

if num_missing > 0:
    # 1. Fill Coordinates (Relative 0,0 = At Event Location)
    # This is correct because x_adj - x_adj = 0
    df_tracking_wide.loc[missing_actor, 'actor_rel_x'] = 0.0
    df_tracking_wide.loc[missing_actor, 'actor_rel_y'] = 0.0
    
    # 2. Fill Velocity (0 = Static/Unknown)
    if 'actor_vx_std' in df_tracking_wide.columns:
        df_tracking_wide.loc[missing_actor, 'actor_vx_std'] = 0.0
        df_tracking_wide.loc[missing_actor, 'actor_vy_std'] = 0.0
    
    # 3. CRITICAL: Set Mask to 1 (Present)
    # The actor EXISTS at this location by definition (the event happened there).
    # mask=0 would tell the model "no player here" which is incorrect.
    df_tracking_wide.loc[missing_actor, 'actor_mask'] = 1.0
    
    # 4. Add 'imputed' flag to distinguish tracked vs. estimated
    # This lets the model know: "Actor is here, but data is estimated"
    df_tracking_wide['actor_is_imputed'] = missing_actor.astype(int)
    
    print(f"\n✓ Missing actor coordinates filled:")
    print(f"  actor_rel_x/y = 0.0 (At Event Location)")
    print(f"  actor_vx/vy   = 0.0 (Static/Unknown)")
    print(f"  actor_mask    = 1.0 (Marked as PRESENT)")
    print(f"  actor_is_imputed = 1 (Flagged as ESTIMATED)")

# Final coverage check
print(f"\n" + "-" * 80)
print(f"Final Actor Coverage After Imputation:")
print(f"-" * 80)
if 'actor_is_imputed' in df_tracking_wide.columns:
    tracked_final = (df_tracking_wide['actor_is_imputed'] == 0).sum()
    imputed_final = (df_tracking_wide['actor_is_imputed'] == 1).sum()
    print(f"  Tracked (Original):  {tracked_final:,} ({tracked_final/len(df_tracking_wide)*100:.1f}%)")
    print(f"  Imputed (Event Loc): {imputed_final:,} ({imputed_final/len(df_tracking_wide)*100:.1f}%)")
else:
    # No imputation needed (all actors were tracked)
    df_tracking_wide['actor_is_imputed'] = 0
    print(f"  All actors tracked: {len(df_tracking_wide):,} (100%)")

print(f"  Total Coverage: 100%")
print(f"\n✓ All events now have actor representation")

STEP 5: PIVOT TO WIDE FORMAT WITH PADDING MASKS (ACTOR-CENTRIC)
Configuration:
  Actor: 1 (event player)
  Max teammates: 5
  Max opponents: 6
  Total players: 12
  Include velocity: True

Pivoting actor (event player)...
  Events with actor: 1,477,009
  Actors tracked: 1031044
  Actors missing (using event coords as fallback): 445965
  Columns created: 5

Pivoting teammates (5 slots)...
  Events with teammate tracking: 1,391,325
  Columns created: 30

Pivoting opponents (6 slots)...
  Events with opponent tracking: 1,407,197
  Columns created: 36

Merging actor + teammates + opponents...
✓ Wide format tracking features: 1,477,009 events
  Total feature columns: 71

--------------------------------------------------------------------------------
Feature Vector Summary:
--------------------------------------------------------------------------------
  Actor (event player) tracked: 69.8% of events
  Average teammates tracked per event: 2.27
  Average opponents tracked per event: 2.74
  A

### 4.10 Merge Tracking Features Back to Events DataFrame

Merge the wide-format tracking features back to the main events DataFrame.

In [15]:
print("=" * 80)
print("STEP 6: MERGE TRACKING FEATURES BACK TO EVENTS")
print("=" * 80)

# Create tracking_key in events if not already present
if 'tracking_key' not in df_events.columns:
    df_events['tracking_key'] = (
        df_events['game_id'].astype(str) + "_" + df_events['sl_event_id'].astype(str)
    )

# Merge tracking features (left join - not all events have tracking)
df_events_with_tracking = df_events.merge(
    df_tracking_wide,
    on='tracking_key',
    how='left'
)

print(f"Events before merge: {len(df_events):,}")
print(f"Events after merge: {len(df_events_with_tracking):,}")
print(f"Events with tracking vectors available pre-merge: {df_tracking_wide.shape[0]:,}")

# Track which rows were directly populated from tracking wide data
joined_from_tracking = df_events_with_tracking['tracking_key'].isin(set(df_tracking_wide['tracking_key']))

# Verify merge
tracking_feature_cols = [c for c in df_tracking_wide.columns if c != 'tracking_key']

# Fill missing tracking vectors with zeros (required for boundary/system rows and non-tracked events)
if tracking_feature_cols:
    df_events_with_tracking[tracking_feature_cols] = df_events_with_tracking[tracking_feature_cols].fillna(0.0)

# Ensure actor_is_imputed is integer-like after fill
if 'actor_is_imputed' in df_events_with_tracking.columns:
    df_events_with_tracking['actor_is_imputed'] = df_events_with_tracking['actor_is_imputed'].round().astype(int)

# Coverage reporting based on actual join, not non-null after fill
events_with_features = int(joined_from_tracking.sum())
print(f"Events populated from tracking join: {events_with_features:,}")
print(f"Events zero-padded (no tracking vector source): {(~joined_from_tracking).sum():,}")

# Check against has_tracking_data flag
if 'has_tracking_data' in df_events_with_tracking.columns:
    expected_tracking = int((df_events_with_tracking['has_tracking_data'] == 1).sum())
    print(f"Expected events with tracking (flag): {expected_tracking:,}")
    coverage = (events_with_features / expected_tracking * 100) if expected_tracking > 0 else 0
    print(f"Join coverage vs flag: {coverage:.1f}%")

print("\n✓ Tracking features merged successfully")
print(f"\nFinal DataFrame shape: {df_events_with_tracking.shape}")
print(f"  Total columns: {len(df_events_with_tracking.columns)}")
print(f"  Tracking feature columns: {len(tracking_feature_cols)}")

# Add distance to net and angle to net calculations for all events
print("\n" + "=" * 80)
print("CALCULATING DISTANCE AND ANGLE TO NET")
print("=" * 80)

# Net position (attacking right towards positive x)
NET_X = 89.0
NET_Y = 0.0

# Calculate distance to net: sqrt((89 - x_adj)^2 + (0 - y_adj)^2)
df_events_with_tracking['distance_to_net_actor'] = np.sqrt(
    (NET_X - df_events_with_tracking['x_adj'])**2 +
    (NET_Y - df_events_with_tracking['y_adj'])**2
)

# Calculate angle to net: arctan(y_adj / (89 - x_adj))
# Returns angle in radians
df_events_with_tracking['angle_to_net_actor'] = np.arctan2(
    df_events_with_tracking['y_adj'],
    NET_X - df_events_with_tracking['x_adj']
)

# Statistics on distance and angle
valid_distance = df_events_with_tracking['distance_to_net_actor'].notna()
valid_angle = df_events_with_tracking['angle_to_net_actor'].notna()

print(f"\nDistance to net statistics:")
print(f"  Valid events: {valid_distance.sum():,}")
if valid_distance.any():
    print(f"  Mean: {df_events_with_tracking.loc[valid_distance, 'distance_to_net_actor'].mean():.2f}")
    print(f"  Median: {df_events_with_tracking.loc[valid_distance, 'distance_to_net_actor'].median():.2f}")
    print(f"  Min: {df_events_with_tracking.loc[valid_distance, 'distance_to_net_actor'].min():.2f}")
    print(f"  Max: {df_events_with_tracking.loc[valid_distance, 'distance_to_net_actor'].max():.2f}")

print(f"\nAngle to net statistics (radians):")
print(f"  Valid events: {valid_angle.sum():,}")
if valid_angle.any():
    print(f"  Mean: {df_events_with_tracking.loc[valid_angle, 'angle_to_net_actor'].mean():.3f}")
    print(f"  Median: {df_events_with_tracking.loc[valid_angle, 'angle_to_net_actor'].median():.3f}")
    print(f"  Min: {df_events_with_tracking.loc[valid_angle, 'angle_to_net_actor'].min():.3f}")
    print(f"  Max: {df_events_with_tracking.loc[valid_angle, 'angle_to_net_actor'].max():.3f}")

print(f"\n✓ Distance and angle to net calculated for all events")

# VALIDATION: Check for events without coordinates
print("\n" + "=" * 80)
print("COORDINATE COVERAGE VALIDATION")
print("=" * 80)

missing_coords = df_events_with_tracking['x_adj'].isna() | df_events_with_tracking['y_adj'].isna()
print(f"\nEvents with missing coordinates: {missing_coords.sum():,}")

if missing_coords.sum() > 0:
    print(f"\nBreakdown by event type:")
    missing_events = df_events_with_tracking[missing_coords]
    event_type_breakdown = missing_events['event_type'].value_counts()
    for event_type, count in event_type_breakdown.items():
        pct = count / missing_coords.sum() * 100
        print(f"  {event_type}: {count:,} ({pct:.1f}%)")

    # Check for non-boundary events without coordinates
    boundary_events = ['whistle', 'goal', 'end_of_period']
    non_boundary_missing = missing_events[~missing_events['event_type'].isin(boundary_events)]

    if len(non_boundary_missing) > 0:
        print(f"\n⚠️  WARNING: {len(non_boundary_missing):,} non-boundary events missing coordinates")
        print(f"   Event types: {non_boundary_missing['event_type'].value_counts().to_dict()}")
    else:
        print(f"\n✓ All missing coordinates are on valid boundary events")
        print(f"   (These rows are retained for continuous-state context)")

print(f"\nDistance_to_net_actor coverage: {df_events_with_tracking['distance_to_net_actor'].notna().sum():,} / {len(df_events_with_tracking):,}")

# Display sample with tracking features
print("-" * 80)
print("Sample event with tracking features:")
print("-" * 80)

sample_with_tracking = df_events_with_tracking[joined_from_tracking].head(1)
if len(sample_with_tracking) > 0:
    info_cols = ['game_id', 'period', 'period_time', 'game_time_sec', 'sequence_id', 'event_type', 'team_id',
                 'is_boundary_event', 'is_end_of_period', 'x_adj', 'y_adj', 'distance_to_net_actor', 'angle_to_net_actor']
    available_info = [c for c in info_cols if c in sample_with_tracking.columns]
    print(sample_with_tracking[available_info].to_string(index=False))

    print("\nTracking features (first 10 columns):")
    print(sample_with_tracking[tracking_feature_cols[:10]].to_string(index=False))

STEP 6: MERGE TRACKING FEATURES BACK TO EVENTS
Events before merge: 1,658,412
Events after merge: 1,658,796
Events with tracking vectors available pre-merge: 1,477,009
Events populated from tracking join: 1,477,337
Events zero-padded (no tracking vector source): 181,459
Expected events with tracking (flag): 1,507,134
Join coverage vs flag: 98.0%

✓ Tracking features merged successfully

Final DataFrame shape: (1658796, 116)
  Total columns: 116
  Tracking feature columns: 72

CALCULATING DISTANCE AND ANGLE TO NET

Distance to net statistics:
  Valid events: 1,633,542
  Mean: 103.12
  Median: 103.23
  Min: 1.02
  Max: 189.59

Angle to net statistics (radians):
  Valid events: 1,633,542
  Mean: -0.001
  Median: -0.001
  Min: -3.139
  Max: 3.141

✓ Distance and angle to net calculated for all events

COORDINATE COVERAGE VALIDATION

Events with missing coordinates: 25,254

Breakdown by event type:
  whistle: 25,254 (100.0%)

✓ All missing coordinates are on valid boundary events
   (These 

### 4.11 Validation & Export

Validate tracking feature quality and save tensor-ready dataset.

In [16]:
print("=" * 80)
print("VALIDATION & QUALITY CHECKS")
print("=" * 80)

# 1. Check mask patterns
print("\n1. MASK VALIDATION:")
print("-" * 80)

actor_mask_col = 'actor_mask'
teammate_mask_cols = [c for c in df_events_with_tracking.columns if 'tm_' in c and '_mask' in c]
opponent_mask_cols = [c for c in df_events_with_tracking.columns if 'opp_' in c and '_mask' in c]

print(f"Actor mask column: 1")
print(f"Teammate mask columns: {len(teammate_mask_cols)}")
print(f"Opponent mask columns: {len(opponent_mask_cols)}")

# For events with tracking joins, check mask distribution
events_with_track = df_events_with_tracking[df_events_with_tracking['tracking_key'].isin(set(df_tracking_wide['tracking_key']))].copy()
if len(events_with_track) > 0:
    actor_present = events_with_track[actor_mask_col].sum()
    tm_counts = events_with_track[teammate_mask_cols].sum(axis=1)
    opp_counts = events_with_track[opponent_mask_cols].sum(axis=1)

    print(f"\nActor (event player) present: {actor_present:.0f} / {len(events_with_track):,} ({actor_present/len(events_with_track)*100:.1f}%)")

    print(f"\nTeammates per event (from masks):")
    print(f"  Mean: {tm_counts.mean():.2f}")
    print(f"  Median: {tm_counts.median():.0f}")
    print(f"  Min: {tm_counts.min():.0f}")
    print(f"  Max: {tm_counts.max():.0f}")

    print(f"\nOpponents per event (from masks):")
    print(f"  Mean: {opp_counts.mean():.2f}")
    print(f"  Median: {opp_counts.median():.0f}")
    print(f"  Min: {opp_counts.min():.0f}")
    print(f"  Max: {opp_counts.max():.0f}")

# Boundary row padding audit
print("\nBoundary event padding audit:")
boundary_mask = df_events_with_tracking['event_type'].isin(['whistle', 'goal', 'end_of_period'])
print(f"  Boundary events: {boundary_mask.sum():,}")
if boundary_mask.any() and actor_mask_col in df_events_with_tracking.columns:
    print(f"  Boundary rows with actor_mask==0: {(df_events_with_tracking.loc[boundary_mask, actor_mask_col] == 0).sum():,}")

# 2. Check for coordinate validity
print("\n2. COORDINATE VALIDATION:")
print("-" * 80)
rel_x_cols = [c for c in df_events_with_tracking.columns if '_rel_x' in c and '_mask' not in c]
rel_y_cols = [c for c in df_events_with_tracking.columns if '_rel_y' in c and '_mask' not in c]

if len(rel_x_cols) > 0 and len(events_with_track) > 0:
    all_rel_x = events_with_track[rel_x_cols].values.flatten()
    all_rel_y = events_with_track[rel_y_cols].values.flatten()

    # Remove padding zeros (but keep actual zeros)
    non_zero_x = all_rel_x[all_rel_x != 0]
    non_zero_y = all_rel_y[all_rel_y != 0]

    if len(non_zero_x) > 0 and len(non_zero_y) > 0:
        print(f"Relative X range: [{non_zero_x.min():.2f}, {non_zero_x.max():.2f}]")
        print(f"Relative Y range: [{non_zero_y.min():.2f}, {non_zero_y.max():.2f}]")

# 3. Check distance validity
print("\n3. DISTANCE VALIDATION:")
print("-" * 80)
dist_cols = [c for c in df_events_with_tracking.columns if '_distance' in c and '_mask' not in c]

if len(dist_cols) > 0 and len(events_with_track) > 0:
    all_dists = events_with_track[dist_cols].values.flatten()
    non_zero_dists = all_dists[all_dists > 0]  # Exclude padding

    if len(non_zero_dists) > 0:
        print(f"Distance range: [{non_zero_dists.min():.2f}, {non_zero_dists.max():.2f}]")
        print(f"Mean distance: {non_zero_dists.mean():.2f}")
        print(f"Median distance: {np.median(non_zero_dists):.2f}")

# 4. Summary statistics
print("\n" + "=" * 80)
print("FINAL SUMMARY")
print("=" * 80)

if INCLUDE_VELOCITY:
    actor_features = 6  # x, y, vx, vy, mask, is_imputed
    context_features = 6  # x, y, vx, vy, dist, mask
    print(f"Features per actor: 6 (x, y, vx, vy, mask, is_imputed)")
    print(f"Features per context player: 6 (x, y, vx, vy, distance, mask)")
else:
    actor_features = 4  # x, y, mask, is_imputed
    context_features = 4  # x, y, dist, mask
    print(f"Features per actor: 4 (x, y, mask, is_imputed)")
    print(f"Features per context player: 4 (x, y, distance, mask)")

print(f"\nPlayer structure:")
print(f"  Actor (event player): 1")
print(f"  Teammates: {MAX_TEAMMATES}")
print(f"  Opponents: {MAX_OPPONENTS}")
print(f"  Total player slots: {1 + MAX_TEAMMATES + MAX_OPPONENTS}")
print(f"Total tracking dimensions: {len(tracking_feature_cols)}")

print(f"\nAdditional event-level features:")
print(f"  • distance_to_net_actor: Euclidean distance to net at (89, 0)")
print(f"  • angle_to_net_actor: Angle to net in radians")
print(f"  • game_time_sec: Monotonic game-clock seconds")
print(f"  • is_boundary_event / is_end_of_period: explicit boundary context flags")

# 5. Save to parquet
print("\n" + "=" * 80)
print("SAVING TENSOR-READY DATA")
print("=" * 80)

output_path = OUTPUT_DIR / "events_with_tracking_features.parquet"
df_events_with_tracking.to_parquet(output_path, index=False)
print(f"✓ Saved to: {output_path}")
print(f"  File contains {len(df_events_with_tracking):,} events")
print(f"  With {len(tracking_feature_cols)} tracking feature columns")
print(f"  Plus distance_to_net_actor and angle_to_net_actor features")

# Build summary JSON
summary_data = {
    'phase': 'Phase 2: Tensor-Ready Data',
    'total_events': int(len(df_events_with_tracking)),
    'events_with_tracking': int(events_with_features),
    'tracking_coverage_pct': float(events_with_features / len(df_events_with_tracking) * 100),
    'tracking_feature_columns': len(tracking_feature_cols),
    'total_dimensions': int(len(tracking_feature_cols)),
    'player_structure': {
        'actor': 1,
        'teammates': int(MAX_TEAMMATES),
        'opponents': int(MAX_OPPONENTS),
        'total': int(1 + MAX_TEAMMATES + MAX_OPPONENTS)
    },
    'include_velocity': bool(INCLUDE_VELOCITY),
    'features_per_actor': int(actor_features),
    'features_per_context_player': int(context_features),
    'coordinate_system': str(tracking_coord_system),
    'avg_actor_tracked': float(avg_actor) if 'avg_actor' in locals() else None,
    'avg_teammates_per_event': float(avg_teammates) if 'avg_teammates' in locals() else None,
    'avg_opponents_per_event': float(avg_opponents) if 'avg_opponents' in locals() else None,
    'has_distance_to_net_actor': True,
    'has_angle_to_net_actor': True,
    'has_game_time_sec': 'game_time_sec' in df_events_with_tracking.columns,
    'has_boundary_flags': all(c in df_events_with_tracking.columns for c in ['is_boundary_event', 'is_end_of_period']),
    'net_coordinates': {'x': 89.0, 'y': 0.0},
    'actor_imputation': {
        'applied': True,
        'imputed_count': int((df_tracking_wide['actor_is_imputed'] == 1).sum()) if 'actor_is_imputed' in df_tracking_wide.columns else 0,
        'imputed_pct': float((df_tracking_wide['actor_is_imputed'] == 1).sum() / len(df_tracking_wide) * 100) if 'actor_is_imputed' in df_tracking_wide.columns else 0.0,
        'method': 'Set rel_x=0, rel_y=0 (at event location), mask=1 (present), is_imputed=1'
    },
    'boundary_padding': {
        'enabled': True,
        'method': 'Rows without tracking-wide match are filled with 0.0 across tracking feature columns'
    },
    'note': 'Continuity-first ordering retained; boundary events are explicitly flagged and kept for downstream rolling-state modeling.'
}

import json
summary_path = OUTPUT_DIR / "tracking_features_summary.json"

with open(summary_path, 'w') as f:
    json.dump(summary_data, f, indent=2)

print(f"✓ Saved summary to: {summary_path}")

print("\n" + "=" * 80)
print("TRACKING DATA INTEGRATION COMPLETE!")
print("=" * 80)

VALIDATION & QUALITY CHECKS

1. MASK VALIDATION:
--------------------------------------------------------------------------------
Actor mask column: 1
Teammate mask columns: 5
Opponent mask columns: 6

Actor (event player) present: 1477337 / 1,477,337 (100.0%)

Teammates per event (from masks):
  Mean: 2.27
  Median: 2
  Min: 0
  Max: 5

Opponents per event (from masks):
  Mean: 2.74
  Median: 3
  Min: 0
  Max: 6

Boundary event padding audit:
  Boundary events: 28,531
  Boundary rows with actor_mask==0: 25,666

2. COORDINATE VALIDATION:
--------------------------------------------------------------------------------
Relative X range: [-193.44, 195.97]
Relative Y range: [-85.99, 85.99]

3. DISTANCE VALIDATION:
--------------------------------------------------------------------------------
Distance range: [0.01, 199.64]
Mean distance: 30.20
Median distance: 28.10

FINAL SUMMARY
Features per actor: 6 (x, y, vx, vy, mask, is_imputed)
Features per context player: 6 (x, y, vx, vy, distance

In [17]:
# Context-aware post-whistle continuity audit
print("=== Post-Whistle Continuity Audit ===")

# Build position arrays for whistle events
whistle_mask = df_events_with_tracking['event_type'] == 'whistle'
whistle_positions = np.where(whistle_mask)[0]

if len(whistle_positions) > 0:
    next_positions = whistle_positions + 1
    valid_next_mask = next_positions < len(df_events_with_tracking)

    valid_whistle_positions = whistle_positions[valid_next_mask]
    valid_next_positions = next_positions[valid_next_mask]

    whistle_rows = df_events_with_tracking.iloc[valid_whistle_positions]
    next_rows = df_events_with_tracking.iloc[valid_next_positions]

    same_seq = (
        (whistle_rows['game_id'].values == next_rows['game_id'].values)
        & (whistle_rows['sequence_id'].values == next_rows['sequence_id'].values)
    )

    next_types = next_rows['event_type'].values
    next_has_player = next_rows['player_id'].notna().values if 'player_id' in next_rows.columns else np.zeros(len(next_rows), dtype=bool)

    # Allowed same-sequence post-whistle continuations from Phase 1 policy
    allowed_post_whistle = np.isin(next_types, ['penalty', 'penaltydrawn']) & next_has_player

    problematic_same_seq = same_seq & (~allowed_post_whistle)

    print(f"Whistles checked: {len(whistle_positions):,}")
    print(f"Whistles with next event in same sequence: {same_seq.sum():,}")
    print(f"Allowed post-whistle penalty continuations: {(same_seq & allowed_post_whistle).sum():,}")

    if problematic_same_seq.sum() == 0:
        print("✓ No unexpected same-sequence post-whistle continuations found.")
    else:
        print(f"⚠ Warning: {problematic_same_seq.sum():,} same-sequence post-whistle continuations are outside allowed policy")

        bad_next_positions = valid_next_positions[problematic_same_seq]
        show_cols = ['game_id', 'period', 'sequence_id', 'period_time', 'event_type', 'player_id']
        show_cols = [c for c in show_cols if c in df_events_with_tracking.columns]
        print(df_events_with_tracking.iloc[bad_next_positions][show_cols].head(20).to_string(index=False))
else:
    print("✓ No whistle events found in dataset.")

=== Post-Whistle Continuity Audit ===
Whistles checked: 25,700
Whistles with next event in same sequence: 427
Allowed post-whistle penalty continuations: 244
⚠ Warning: 183 same-sequence post-whistle continuations are outside allowed policy
                             game_id  period  sequence_id  period_time event_type                            player_id
00f1ee7c-b2e4-3fee-b8ba-37158dc3166d       4           52       299.17    whistle c05db9ee-e2c8-5bba-3a20-cee81b8b04c1
00f1ee7c-b2e4-3fee-b8ba-37158dc3166d       4           52       299.17    whistle c05db9ee-e2c8-5bba-3a20-cee81b8b04c1
00f1ee7c-b2e4-3fee-b8ba-37158dc3166d       4           52       299.17    whistle c05db9ee-e2c8-5bba-3a20-cee81b8b04c1
07c3dfdb-28d2-df56-d0f5-a20f2016aee4       1           20      1041.03    whistle 5ad461c4-01e6-3090-18c9-0aa215276809
07c3dfdb-28d2-df56-d0f5-a20f2016aee4       1           20      1041.03    whistle 5ad461c4-01e6-3090-18c9-0aa215276809
07c3dfdb-28d2-df56-d0f5-a20f2016aee4       1 

---

## Tracking Data Integration Summary

✅ **Completed Pipeline:**

1. **Data Loading & Cleaning** - Loaded ~13.5M tracking rows, removed None player_ids, deduplicated entries
2. **Player Count Analysis** - Removed ~3.2M None player_id rows and duplicate entries
3. **Coordinate Verification** - Determined tracking uses RAW coordinates (x, y not x_adj, y_adj)
4. **Coordinate Transformation** - Applied flip transformation to standardize to adjusted coordinate system
5. **Role Identification** - Classified players as teammates vs opponents based on acting team
6. **Distance Calculation** - Computed relative positions and Euclidean distance from event location
7. **Player Ranking** - Sorted players by proximity to event (closest = rank 1)
8. **Wide Pivot with Padding Masks** - Transformed long format → fixed-size tensor-ready vectors

**Output Feature Vector Structure (Actor-Centric, Skaters Only):**
- **Actor (1 slot)**: `actor_rel_x, actor_rel_y, actor_vx_std, actor_vy_std, actor_mask`
  - When tracked (actor_is_imputed=0): actual position relative to event, mask=1
  - When imputed (actor_is_imputed=1): fallback to event location (rel_x=0, rel_y=0), mask=1
  - **Mask = 1 always**: Actor is physically present at event location (100% coverage)
- **Teammates (5 slots)**: `tm_1_rel_x, tm_1_rel_y, tm_1_vx_std, tm_1_vy_std, tm_1_distance, tm_1_mask, ...`
- **Opponents (6 slots)**: `opp_1_rel_x, opp_1_rel_y, opp_1_vx_std, opp_1_vy_std, opp_1_distance, opp_1_mask, ...`
- **Total dimensions**: 72 (1 actor × 6 features [including imputed flag] + 5 teammates × 6 features + 6 opponents × 6 features)
- **Player Structure**: Actor is always the event player (100% coverage: 69% tracked + 31% imputed), teammates exclude the actor, ranked by proximity

**Event-Level Features:**
- **distance_to_net**: Euclidean distance from event location to net at (89, 0)
  - Formula: $\sqrt{(89 - x_{adj})^2 + (0 - y_{adj})^2}$
- **angle_to_net**: Angle from event location to net in radians
  - Formula: $\arctan\left(\frac{y_{adj}}{89 - x_{adj}}\right)$

**Goalie Tracking Decision:**
- Skipped due to insufficient coverage (4.5% overall, 7.9% of shots)
- Focusing on skater positions only for reliable modeling

**Padding Strategy:**
- Missing positions filled with `0.0`
- Binary mask: `1` = player present, `0` = padded slot
- **Actor mask always = 1** (actor is physically present at event location by definition)
- Transformer will use masks in `src_key_padding_mask` to ignore padded slots

**Actor Imputation:**
- Events without tracked actor (31%): set rel_x=0, rel_y=0 (at event location)
- Actor mask set to 1 (not 0) because actor IS present at the event
- `actor_is_imputed` flag distinguishes tracked (0) vs. imputed (1) actors
- Result: 100% actor coverage (69% tracked + 31% imputed at event location)

**Next Steps:**
- ✅ Sequence terminators filtered (whistles/goals have no acting_team_id)
- ✅ Actor imputation applied during pivot (100% coverage achieved)
- Proceed to Phase 3: Sequence assembly and tensor encoding

---